# CS781 Phase 3. Heavier regularization for the efficient teacher

I kept the original efficient model and its initial weights, and increased backbone dropout from 0.10 to 0.20, head dropout from 0.20 to 0.40 and AdamW weight decay from 0.01 to 0.05. These settings changed together. Dev macro F1 rose from 0.4436 to 0.4510, while the CodaBench score fell from 0.4408 to 0.4249.

| Run | Best epoch | Epochs completed | Dev macro F1 | CodaBench macro F1 | Submission ID |
| --- | ---: | ---: | ---: | ---: | ---: |
| Heavier regularization | 16 | 18 | 0.4510 | 0.4249 | 994303 |


## 1. Inputs and environment

1. Start a Kaggle session with two T4 GPUs. Use the session's PyTorch installation.
2. Attach `prep_002_mixed16k` with its tokenizer and manifest. Attach the original
   [Phase 1 run](../phase1/phase1_efficient_500m_continuation.ipynb) `efficient_002_mixed217m_500m` with `best_model.pt`
   and its source files. Set `PREP_INPUT` and `EFFICIENT_INPUT` if automatic
   discovery is ambiguous.
3. Set `RR_DATA_INPUT` to the official data folder if it is already available.
   For the pinned download, enable Internet and provide the Kaggle secret `HF_TOKEN`.
4. Keep `STAGE = "teacher"`. Use a fresh run directory or a matching heavier-regularization
   backup. This setup needs its own passing preflight because the dropout settings differ.


In [1]:
%pip install -q huggingface_hub pandas scikit-learn matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import hashlib
import importlib.metadata
import json
from pathlib import Path
import shutil
import sys
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display, FileLink

# One fresh full-teacher experiment. Keep the tag unchanged only for an exact resume.
STAGE = 'teacher'
EXPERIMENT_TAG = 'heavyreg_001'
TEACHER_ADAPTATION = 'full'  # full teacher; no automatic freezing

PREP_INPUT = None
EFFICIENT_INPUT = None
RR_DATA_INPUT = None
TEACHER_CACHE_INPUT = None  # unused: supervised teacher only
PREFLIGHT_INPUT = None      # old dropout proofs cannot authorize this configuration
RESUME_INPUT = None         # optional exact stage backup ZIP/folder from another session

# Operational limits per invocation. Check your ACTUAL remaining GPU quota first.
# These can be changed on resume; model settings and epoch caps stay frozen.
RUN_BUDGET_HOURS = {'teacher': 6, 'student_baseline': 2.0, 'student_kd': 5.0}
SAVE_EVERY_MINUTES = 5.0
LOG_EVERY_UPDATES = 10
STOP_AFTER_UPDATES = None   # optional deliberate pause for a resume check

ROOT = Path('/kaggle/working/cs781_phase3')
SRC = Path('/kaggle/working/cs781_phase3_efficient_rr_heavy_src')
for folder in (ROOT, SRC, ROOT / 'runs'):
    folder.mkdir(parents=True, exist_ok=True)
if STAGE != 'teacher':
    raise ValueError('This notebook runs one supervised teacher experiment.')
if TEACHER_ADAPTATION != 'full':
    raise ValueError('Keep full adaptation for this teacher comparison.')
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle T4 x2.')
if any(torch.cuda.memory_allocated(i) > 256 * 2**20 for i in range(2)):
    raise RuntimeError('Restart the kernel: another GPU model is still allocated.')
torch.set_num_threads(2)
COMMON = {
    'seed':42, 'window_sentences':128, 'overlap_sentences':32,
    'chunk_tokens':256, 'chunk_microbatch':8, 'global_window_batch':2, 'accumulation_steps':2,
    'gradient_checkpointing':True, 'projection_dim':256, 'context_hidden':128, 'head_dropout':0.4,
    'backbone_dropout':0.2,
    'pooling':'mean_last', 'student_layers':[0,7,15],
    'teacher_lr':1e-5, 'student_lr':2e-5, 'head_lr':3e-4, 'crf_lr':1e-3,
    'weight_decay':0.05, 'gradient_clip':1.0, 'class_weight_cap':5.0,
    'aux_ce_coefficient':0.5, 'temperature':2.0, 'kd_weight':0.4,
    'learning_steps':40, 'teacher_training_hours':RUN_BUDGET_HOURS['teacher'],
    'student_training_hours':RUN_BUDGET_HOURS['student_kd'],
}
TRAINING = {
    'minimum_epochs':5, 'patience':5, 'early_stop_min_delta':1e-4,
    'warmup_fraction':0.05, 'minimum_lr_fraction':0.01, 'train_metrics_every_epochs':3,
}
EPOCH_CAPS = {'teacher':35, 'student_baseline':35, 'student_kd':35}
RUN_IDS = {'teacher':'efficient_rr_teacher_' + EXPERIMENT_TAG,
    'student_baseline':'efficient_rr_student_gold_' + EXPERIMENT_TAG,
    'student_kd':'efficient_rr_student_kd_' + EXPERIMENT_TAG}
print('Stage:', STAGE, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])
print('Working free GiB:', round(shutil.disk_usage(ROOT).free / 2**30, 2))
print('Per-invocation training-hour budgets:', RUN_BUDGET_HOURS)

print('Regularization:', {k:COMMON[k] for k in ('backbone_dropout','head_dropout','weight_decay')})


Stage: teacher | GPUs: ['Tesla T4', 'Tesla T4']
Working free GiB: 15.77
Per-invocation training-hour budgets: {'teacher': 6, 'student_baseline': 2.0, 'student_kd': 5.0}
Regularization: {'backbone_dropout': 0.2, 'head_dropout': 0.4, 'weight_decay': 0.05}


## 2. Efficient model and dropout configuration

These cells write the original Phase 1 tokenizer and backbone, data preparation, role model, training helpers and course evaluator into the Kaggle working directory. The run checks these sources against its saved provenance.


In [3]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/phase1_utils.py
"""Small file and JSON helpers used by the Phase-1 notebooks."""
import hashlib
import json
import os
import pickle
import zipfile
from pathlib import Path


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def atomic_pickle(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "wb") as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def checked_json(path, value):
    path = Path(path)
    if path.exists() and read_json(path) != value:
        raise ValueError(f"Saved settings differ: {path}. Use a new experiment ID.")
    atomic_json(path, value)


def write_source(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    raw = content.encode("utf-8") if isinstance(content, str) else content
    if path.exists():
        if path.read_bytes() != raw:
            raise ValueError(f"Saved source differs: {path}. Keep its code or use a new experiment ID.")
        return
    path.write_bytes(raw)


def make_bundle(folder, output_path, archive_root):
    folder, output_path, archive_root = Path(folder), Path(output_path), Path(archive_root)
    temporary = output_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(archive_root))
    os.replace(temporary, output_path)
    return output_path


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/phase1_utils.py


In [4]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/a1_tokenizer.py
"""The assignment byte-BPE algorithm, with checkpoints between merges.

Pair counting, lexicographic tie breaking, non-overlapping replacements and
merge-order encoding are unchanged from the completed Phase-1/A3 tokenizer.
"""
import pickle
import re
import time
from collections import Counter
from pathlib import Path
from phase1_utils import atomic_pickle, sha256_file, text_hash


class A1BPETokenizer:
    def __init__(self):
        specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.vocab = [s.encode("utf-8") for s in specials]
        self.vocab += [bytes([b]) for b in range(256)]
        self.boundary_id = len(self.vocab)
        self.vocab.append(b"")
        self.names = [s.encode("utf-8") for s in specials]
        self.names += [bytes([b]) for b in range(256)]
        self.names.append(b"</w>")
        self.merges = {}
        self.vocab_size = len(self.vocab)

    def fit(self, text, vocab_size, checkpoint_path=None, checkpoint_merges=100,
            stop_after_merges=None):
        if vocab_size < 261:
            raise ValueError("Vocabulary must accommodate bytes and reserved tokens")
        signature = {"text_sha256": text_hash(text), "vocab_size": vocab_size,
                     "algorithm_source_sha256": sha256_file(__file__)}
        checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        elapsed_before = 0.0
        if checkpoint_path is not None and checkpoint_path.exists():
            with open(checkpoint_path, "rb") as file:
                saved = pickle.load(file)
            if saved["signature"] != signature:
                raise ValueError("BPE checkpoint settings or source differ")
            self.__dict__.update(saved["tokenizer_state"])
            words = saved["words"]
            elapsed_before = saved["elapsed_seconds"]
            print("Resuming BPE vocabulary:", self.vocab_size, flush=True)
        else:
            if self.merges:
                raise ValueError("Use a fresh tokenizer or its saved fitting checkpoint")
            words = Counter(tuple(b + 4 for b in piece.encode("utf-8")) + (self.boundary_id,)
                            for piece in re.findall(r"\S+|\s+", text))
        started = time.monotonic()
        last_saved = started
        merges_this_call = 0
        commit_in_progress = False

        def snapshot():
            if checkpoint_path is not None:
                atomic_pickle(checkpoint_path, {"signature": signature,
                    "tokenizer_state": self.__dict__, "words": words,
                    "elapsed_seconds": elapsed_before + time.monotonic() - started})

        try:
            while len(self.vocab) < vocab_size:
                pair_counts = Counter()
                for symbols, frequency in words.items():
                    for pair in zip(symbols, symbols[1:]):
                        pair_counts[pair] += frequency
                if not pair_counts:
                    break
                pair = min(pair_counts, key=lambda p: (-pair_counts[p], self.names[p[0]], self.names[p[1]]))
                if pair_counts[pair] < 2:
                    break
                new_id = len(self.vocab)
                updated = Counter()
                for symbols, frequency in words.items():
                    result = []
                    j = 0
                    while j < len(symbols):
                        if j + 1 < len(symbols) and symbols[j:j + 2] == pair:
                            result.append(new_id)
                            j += 2
                        else:
                            result.append(symbols[j])
                            j += 1
                    updated[tuple(result)] += frequency
                # Commit the whole merge only after every word has been replaced.
                commit_in_progress = True
                self.vocab.append(self.vocab[pair[0]] + self.vocab[pair[1]])
                self.names.append(self.names[pair[0]] + self.names[pair[1]])
                self.merges[pair] = new_id
                self.vocab_size = len(self.vocab)
                words = updated
                commit_in_progress = False
                merges_this_call += 1
                if len(self.merges) % checkpoint_merges == 0 or time.monotonic() - last_saved >= 60:
                    snapshot()
                    print(f"BPE vocabulary {self.vocab_size:,}/{vocab_size:,} | "
                          f"elapsed {(elapsed_before + time.monotonic() - started)/60:.1f} min", flush=True)
                    last_saved = time.monotonic()
                if stop_after_merges is not None and merges_this_call >= stop_after_merges:
                    break
        finally:
            if not commit_in_progress:
                snapshot()
        return self

    def encode(self, text):
        result = []
        for piece in re.findall(r"\S+|\s+", text):
            symbols = [b + 4 for b in piece.encode("utf-8")] + [self.boundary_id]
            while len(symbols) > 1:
                candidates = [(self.merges[(a, b)], (a, b))
                              for a, b in zip(symbols, symbols[1:]) if (a, b) in self.merges]
                if not candidates:
                    break
                new_id, pair = min(candidates)
                merged = []
                j = 0
                while j < len(symbols):
                    if j + 1 < len(symbols) and (symbols[j], symbols[j + 1]) == pair:
                        merged.append(new_id)
                        j += 2
                    else:
                        merged.append(symbols[j])
                        j += 1
                symbols = merged
            result.extend(symbols)
        return result

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8")


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/a1_tokenizer.py


In [5]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/architecture_models.py
"""Efficient causal LM: assignment GQA with actual sliding-window attention.

Course Module 2.3 pages 155-156, 167-170 and 180-185.
A3 Q2 supplies the KV-head sharing algorithm. SDPA Boolean True means allowed.
Training uses next-token targets. KV caches are inference-only.
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    architecture: str = "efficient"
    vocab_size: int = 16000
    d_model: int = 1024
    n_heads: int = 16
    n_layers: int = 16
    n_kv_heads: int = 4
    d_ff: int = 4864
    max_seq_len: int = 2560
    dropout: float = 0.1
    window_size: int = 512
    query_chunk_size: int = 128

    def __post_init__(self):
        if self.architecture != "efficient":
            raise ValueError("This notebook implements the efficient architecture only")
        assert self.d_model % self.n_heads == 0 and self.n_heads % self.n_kv_heads == 0
        assert min(self.vocab_size - 4, self.n_layers, self.d_ff, self.max_seq_len,
                   self.window_size, self.query_chunk_size) > 0
        assert 0 <= self.dropout < 1


def configuration(architecture="efficient"):
    return ModelConfig(architecture=architecture)



def positions(max_len, width):
    p = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, width, 2).float() * (-math.log(10000.0) / width))
    result = torch.zeros(max_len, width)
    result[:, 0::2] = torch.sin(p * frequency)
    result[:, 1::2] = torch.cos(p * frequency)
    return result


class Attention(nn.Module):
    def __init__(self, cfg, kv_heads=None):
        super().__init__()
        self.heads = cfg.n_heads
        self.kv_heads = cfg.n_heads if kv_heads is None else kv_heads
        assert cfg.d_model % self.heads == 0 and self.heads % self.kv_heads == 0
        self.head_dim = cfg.d_model // self.heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def query(self, x):
        b, t, _ = x.shape
        return self.q_proj(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)

    def keys_values(self, x):
        b, t, _ = x.shape
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        return k, v

    def attend(self, q, k, v, allowed=None, causal=False):
        # A3 Q2: repeat each KV head for its group of query heads.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k, v = k.repeat_interleave(repeats, dim=1), v.repeat_interleave(repeats, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0, scale=1 / math.sqrt(self.head_dim))
        b, _, t, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(b, t, -1)
        return self.resid_dropout(self.out_proj(out))

    def forward(self, x, memory=None, allowed=None, causal=False):
        k, v = self.keys_values(x if memory is None else memory)
        return self.attend(self.query(x), k, v, allowed, causal)

    def local(self, x, window, chunk, lengths=None):
        # Query tiles see only nearby keys. No full T-by-T score matrix.
        q = self.query(x)
        k, v = self.keys_values(x)
        t = x.shape[1]
        pieces = []
        for start in range(0, t, chunk):
            stop = min(t, start + chunk)
            first = max(0, start - window + 1)
            qi = torch.arange(start, stop, device=x.device)
            ki = torch.arange(first, stop, device=x.device)
            allowed = (ki[None, :] <= qi[:, None]) & (ki[None, :] > qi[:, None] - window)
            allowed = allowed[None, None, :, :]
            if lengths is not None:
                allowed = allowed & (ki[None, :] < lengths[:, None])[:, None, None, :]
            pieces.append(self.attend(q[:, :, start:stop], k[:, :, first:stop], v[:, :, first:stop], allowed))
        return torch.cat(pieces, dim=1)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
                                 nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout))

    def forward(self, x):
        return self.net(x)


class EfficientBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg, cfg.n_kv_heads), FeedForward(cfg)
        self.window, self.chunk = cfg.window_size, cfg.query_chunk_size

    def forward(self, x, lengths=None):
        x = x + self.attn.local(self.ln1(x), self.window, self.chunk, lengths)
        return x + self.ffn(self.ln2(x))


class BaseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", positions(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def initialize(self):
        for p in self.parameters():
            if p.ndim > 1:
                nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def embed(self, ids, offset=None):
        t = ids.shape[1]
        p = torch.arange(t, device=ids.device)[None, :]
        if offset is not None:
            p = p + offset[:, None]
        x = self.tok_emb(ids) * math.sqrt(self.cfg.d_model) + self.pos_emb[p]
        return x


class EfficientLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.blocks = nn.ModuleList([EfficientBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, ids, lengths=None):
        if ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Input exceeds context budget")
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, lengths)
        return self.head(self.final_norm(x))

    def start_generation(self, prompt, lengths):
        x = self.embed(prompt)
        layers = []
        keep = self.cfg.window_size - 1
        indices = lengths[:, None] - keep + torch.arange(keep, device=prompt.device)[None, :]
        gather = indices.clamp_min(0)[:, None, :, None]
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            shape = (prompt.shape[0], k.shape[1], keep, k.shape[-1])
            layers.append({"k": torch.gather(k, 2, gather.expand(shape)),
                           "v": torch.gather(v, 2, gather.expand(shape))})
            x = x + block.attn.local(h, block.window, block.chunk, lengths)
            x = x + block.ffn(block.ln2(x))
        last = self.final_norm(x)[torch.arange(x.shape[0], device=x.device), lengths - 1]
        cache = {"layers": layers, "lengths": lengths, "next_positions": lengths.clone(),
                 "key_positions": indices}
        return self.head(last), cache

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"])
        key_positions = torch.cat([cache["key_positions"], cache["next_positions"][:, None]], dim=1)
        allowed = (key_positions >= 0)[:, None, None, :]
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            k, v = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
            layer["k"], layer["v"] = k[:, :, 1:], v[:, :, 1:]
        cache["key_positions"] = key_positions[:, 1:]
        cache["next_positions"] = cache["next_positions"] + 1
        return self.head(self.final_norm(x))[:, -1], cache


def select_cache(cache, indices):
    """Remove finished rows only when the active batch shrinks."""
    if isinstance(cache, torch.Tensor):
        return cache.index_select(0, indices)
    if isinstance(cache, dict):
        return {k: select_cache(v, indices) for k, v in cache.items()}
    if isinstance(cache, list):
        return [select_cache(v, indices) for v in cache]
    return cache


class ModelWithLoss(nn.Module):
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm, self.use_amp = lm, use_amp

    def forward(self, x, y, prefixes):
        # DataParallel scatters this per-example tensor along with x and y.
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits, labels = logits_and_labels(self.lm, x, y, 0)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), labels.reshape(-1))
        return loss.unsqueeze(0)


def build_model(cfg):
    return EfficientLM(cfg)


def logits_and_labels(lm, x, y, prefix_length=0):
    # y is the token immediately after each input position in x.
    return lm(x), y


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/architecture_models.py


In [6]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_data.py
import hashlib
import json
import math
import pickle
import re
import shutil
import time
import zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch
ROOT = Path('/kaggle/working/cs781_phase3')
INPUT_ROOTS = [Path('/kaggle/input')]

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for part in iter(lambda: f.read(2**20), b''):
            h.update(part)
    return h.hexdigest()

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def read_json(path):
    def unique_pairs(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f'Duplicate JSON key: {key!r}')
            result[key] = value
        return result
    def reject_constant(value):
        raise ValueError(f'Invalid JSON constant: {value}')
    return json.loads(Path(path).read_text(encoding='utf-8'),
                      object_pairs_hook=unique_pairs, parse_constant=reject_constant)

def find_input(filename, supplied=None, hint=None, required=True):
    archives = []
    if supplied is not None:
        p = Path(supplied)
        if p.is_file() and p.suffix.lower() == '.zip':
            candidates, archives = [], [p]
        else:
            candidates = [p] if p.is_file() and p.name == filename else list(p.rglob(filename))
            archives = list(p.rglob('*.zip')) if p.is_dir() else []
    else:
        candidates = [p for root in INPUT_ROOTS if root.exists() for p in root.rglob(filename)]
        if hint is not None:
            candidates = [p for p in candidates if hint in str(p)]
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        if supplied is None:
            archives = [p for root in INPUT_ROOTS if root.exists() for p in root.rglob('*.zip')]
        archive_matches = []
        for archive in sorted(set(archives)):
            try:
                with zipfile.ZipFile(archive) as z:
                    for info in z.infolist():
                        if Path(info.filename).name == filename and not info.is_dir():
                            if hint is None or hint in str(archive) + '/' + info.filename:
                                archive_matches.append((archive, info.filename))
            except zipfile.BadZipFile:
                continue
        if len(archive_matches) > 1:
            raise ValueError('Several archive candidates; set an exact input path: ' + str(archive_matches))
        if archive_matches:
            archive, member = archive_matches[0]
            dest = ROOT / 'extracted_inputs' / hashlib.sha256(str(archive.resolve()).encode()).hexdigest()[:12]
            dest.mkdir(parents=True, exist_ok=True)
            metadata_path = dest / 'extraction_record.json'
            records = read_json(metadata_path) if metadata_path.exists() else {}
            with zipfile.ZipFile(archive) as z:
                wanted = [member]
                sibling = str(Path(member).parent / 'cache_manifest.json') if filename == 'tokenizer_state.pkl' else str(Path(member).parent / 'sources/architecture_models.py')
                if sibling in z.namelist():
                    wanted.append(sibling)
                for name in wanted:
                    info = z.getinfo(name)
                    target = (dest / name).resolve()
                    if not target.is_relative_to(dest.resolve()):
                        raise ValueError('Unsafe path inside attached ZIP.')
                    record = {'archive_bytes': archive.stat().st_size, 'archive_mtime_ns': archive.stat().st_mtime_ns,
                              'member_crc': info.CRC, 'member_bytes': info.file_size}
                    if target.exists() and target.stat().st_size == info.file_size and records.get(name) == record:
                        continue
                    if shutil.disk_usage(ROOT).free < info.file_size + 256 * 2**20:
                        raise RuntimeError('Insufficient working disk to extract this selected Phase 1 file.')
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_suffix(target.suffix + '.tmp')
                    with z.open(name) as src, open(temporary, 'wb') as dst:
                        shutil.copyfileobj(src, dst, length=2**20)
                    temporary.replace(target)
                    records[name] = record
                    print('Extracted selected input:', archive.name, '->', name)
            write_json(metadata_path, records)
            return dest / member
    if not candidates and not required:
        return None
    if not candidates:
        raise FileNotFoundError(f'Cannot find {filename}. Attach its extracted input folder or set an exact path.')
    raise ValueError(f'Several candidates for {filename}; set an exact path:\n' + '\n'.join(map(str, candidates)))

def official_labels(obj):
    if isinstance(obj, list):
        result = obj
    elif isinstance(obj, dict) and obj:
        if all(isinstance(v, int) and not isinstance(v, bool) for v in obj.values()):
            if set(obj.values()) != set(range(13)):
                raise ValueError('Expected label IDs 0..12 in label_vocab.json')
            result = [label for label, idx in sorted(obj.items(), key=lambda pair: pair[1])]
        elif all(isinstance(v, str) for v in obj.values()) and all(str(k).isdigit() for k in obj):
            if {int(k) for k in obj} != set(range(13)):
                raise ValueError('Expected label IDs 0..12 in label_vocab.json')
            result = [obj[str(i)] if str(i) in obj else obj[i] for i in range(13)]
        else:
            raise ValueError('Unsupported label_vocab.json mapping; inspect its preview below.')
    else:
        raise ValueError('Unsupported label vocabulary structure.')
    if len(result) != 13 or len(set(result)) != 13 or any(not isinstance(v, str) or not v.strip() for v in result):
        raise ValueError('Expected 13 unique nonempty label strings.')
    return result

def normalize_id(raw_id):
    if not isinstance(raw_id, str) or not raw_id.strip():
        raise ValueError('Document IDs must be nonempty strings.')
    return raw_id[:-4] if raw_id.endswith('.txt') else raw_id

def parse_documents(raw, domain, split, filename):
    if isinstance(raw, list):
        items = [(None, record) for record in raw]
    elif isinstance(raw, dict) and 'text' in raw and 'id' in raw:
        items = [(None, raw)]
    elif isinstance(raw, dict):
        items = list(raw.items())
    else:
        raise ValueError(f'{filename}: unsupported top-level structure.')
    result, seen = [], set()
    for key, record in items:
        if split == 'test' and key is not None and isinstance(record, list):
            record = {'id': key, 'text': record}
        if not isinstance(record, dict):
            raise ValueError(f'{filename}: expected records with id/text/labels; inspect the JSON preview.')
        raw_id = record.get('id', key)
        doc_id = normalize_id(raw_id)
        if key is not None and normalize_id(key) != doc_id:
            raise ValueError(f'{filename}: mapping key and record id disagree.')
        if doc_id in seen:
            raise ValueError(f'{filename}: duplicate normalized document ID {doc_id!r}')
        seen.add(doc_id)
        text = record.get('text')
        if not isinstance(text, list) or any(not isinstance(s, str) for s in text):
            raise ValueError(f'{filename}/{doc_id}: text must be an ordered list of strings.')
        labels = record.get('labels') if split != 'test' else None
        if split != 'test':
            if not isinstance(labels, list) or len(labels) != len(text):
                raise ValueError(f'{filename}/{doc_id}: sentence/label count mismatch.')
            if any(not isinstance(label, str) or label not in LABEL_TO_ID for label in labels):
                raise ValueError(f'{filename}/{doc_id}: unknown or non-string role label.')
        result.append({'domain': domain, 'split': split, 'file': filename, 'raw_id': raw_id,
                       'id': doc_id, 'text': text, 'labels': labels,
                       'test_labels_field_present': split == 'test' and 'labels' in record})
    return result

def sentence_windows(n, width, overlap):
    if not 0 <= overlap < width:
        raise ValueError('Require 0 <= overlap < width.')
    if n == 0:
        return []
    starts = [0]
    while starts[-1] + width < n:
        starts.append(starts[-1] + width - overlap)
    ends = [min(s + width, n) for s in starts]
    boundaries = [0] + [(ends[i-1] + starts[i]) // 2 for i in range(1, len(starts))] + [n]
    windows = [{'start': s, 'end': e, 'owned_start': boundaries[i], 'owned_end': boundaries[i+1]}
               for i, (s, e) in enumerate(zip(starts, ends))]
    ownership = np.zeros(n, dtype=np.int32)
    for w in windows:
        assert w['start'] <= w['owned_start'] <= w['owned_end'] <= w['end']
        ownership[w['owned_start']:w['owned_end']] += 1
    assert (ownership == 1).all()
    return windows

def split_chunks(ids, cap):
    if cap < 3:
        raise ValueError('Chunk cap must leave space for content plus two specials.')
    parts = [ids[i:i + cap - 2] for i in range(0, len(ids), cap - 2)] or [[]]
    return [[BOS_ID] + part + [EOS_ID] for part in parts]

PAD_ID, BOS_ID, EOS_ID = 0, 2, 3


def prepare_rr_data(destination, expected_hashes, revision, supplied=None):
    """Use verified local RR bytes when available; otherwise download seven files."""
    destination = Path(destination)
    relatives = {name: ('Test/' if name.endswith('_test.json') else 'Train/') + name
                 for name in expected_hashes}
    candidates = []
    if supplied is not None:
        p = Path(supplied)
        if not p.is_dir():
            raise ValueError('RR_DATA_INPUT must be an extracted data folder.')
        candidates = [p] + [f.parent.parent for f in p.rglob('CL_train.json')]
    else:
        candidates = [destination, Path('/kaggle/working/cs781_phase3_audit/official_data/Phase_3_RR')]
        candidates += [p.parent.parent for root in INPUT_ROOTS if root.exists()
                       for p in root.rglob('CL_train.json')]
    for candidate in candidates:
        if all((candidate / rel).exists() for rel in relatives.values()):
            if all(file_hash(candidate / relatives[name]) == digest for name, digest in expected_hashes.items()):
                return candidate
            if supplied is not None:
                raise ValueError('Selected RR data differs from the audited official revision.')
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import disable_progress_bars
    from kaggle_secrets import UserSecretsClient
    disable_progress_bars()
    try:
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        raise RuntimeError('Enable Internet and the Kaggle secret HF_TOKEN, or attach verified RR files.') from None
    try:
        for name, rel in relatives.items():
            path = hf_hub_download(repo_id='Exploration-Lab/CS-781-Capstone', repo_type='dataset',
                filename='Phase_3_RR/' + rel, revision=revision, token=token,
                local_dir=str(destination.parent))
            if file_hash(path) != expected_hashes[name]:
                raise ValueError('Downloaded RR file differs from the audited hash: ' + name)
            print('RR data ready:', name, flush=True)
    except Exception:
        raise RuntimeError('RR download failed. Check Internet and HF_TOKEN access to the official dataset.') from None
    finally:
        token = None
    return destination


def load_rr_documents(folder):
    global LABEL_TO_ID
    folder = Path(folder)
    labels = official_labels(read_json(folder / 'Train/label_vocab.json'))
    LABEL_TO_ID = {name: i for i, name in enumerate(labels)}
    documents = []
    for domain in ('CL', 'IT'):
        for split in ('train', 'dev', 'test'):
            name = domain + '_' + split + '.json'
            raw = read_json(folder / ('Test' if split == 'test' else 'Train') / name)
            documents.extend(parse_documents(raw, domain, split, name))
    for split in ('train', 'dev', 'test'):
        ids = [d['id'] for d in documents if d['split'] == split]
        if len(ids) != len(set(ids)):
            raise ValueError('Duplicate normalized IDs across domains in ' + split)
    for a, b in (('train', 'dev'), ('train', 'test'), ('dev', 'test')):
        if {d['id'] for d in documents if d['split'] == a} & {d['id'] for d in documents if d['split'] == b}:
            raise ValueError('Document IDs overlap across splits.')
    return labels, documents


def load_tokenizer(path):
    from a1_tokenizer import A1BPETokenizer
    path = Path(path)
    manifest = read_json(path.parent / 'cache_manifest.json')
    if file_hash(path) != manifest['tokenizer_state_sha256']:
        raise ValueError('Tokenizer bytes differ from the preparation manifest.')
    with open(path, 'rb') as f:
        saved = pickle.load(f)
    if saved['signature'] != manifest['tokenizer_signature']:
        raise ValueError('Tokenizer signatures differ.')
    tokenizer = A1BPETokenizer()
    tokenizer.__dict__.update(saved['state'])
    assert tokenizer.vocab[:4] == [b'<pad>', b'<unk>', b'<s>', b'</s>']
    return tokenizer, saved['signature'], manifest


def tokenize_documents(documents, tokenizer, fingerprint, cache_dir):
    cache = Path(cache_dir) / (fingerprint + '.pkl')
    if cache.exists():
        with open(cache, 'rb') as f:
            saved = pickle.load(f)
        if saved['fingerprint'] != fingerprint or len(saved['tokens']) != len(documents):
            raise ValueError('RR token cache identity differs.')
        for doc, tokens, identity in zip(documents, saved['tokens'], saved['identities']):
            if identity != [doc['file'], doc['raw_id']] or len(tokens) != len(doc['text']):
                raise ValueError('RR token cache sentence alignment differs.')
            doc['token_ids'] = tokens
        print('Loaded verified RR token cache.', flush=True)
        return
    from functools import lru_cache
    @lru_cache(maxsize=100000)
    def piece_ids(piece):
        return tuple(tokenizer.encode(piece))
    started = last = time.monotonic()
    for i, doc in enumerate(documents):
        doc['token_ids'] = [[idx for p in re.findall(r'\S+|\s+', sentence) for idx in piece_ids(p)]
                            for sentence in doc['text']]
        if i == 0 or time.monotonic() - last > 30:
            print('Tokenized RR documents:', i + 1, '/', len(documents), flush=True)
            last = time.monotonic()
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache.with_suffix('.tmp'), 'wb') as f:
        pickle.dump({'fingerprint': fingerprint, 'tokens': [d['token_ids'] for d in documents],
                     'identities': [[d['file'], d['raw_id']] for d in documents]}, f, protocol=pickle.HIGHEST_PROTOCOL)
    cache.with_suffix('.tmp').replace(cache)
    print('Tokenization minutes:', round((time.monotonic() - started) / 60, 2), flush=True)


def make_windows(documents, width, overlap, split):
    result = []
    for di, doc in enumerate(documents):
        if doc['split'] != split:
            continue
        for w in sentence_windows(len(doc['text']), width, overlap):
            result.append(dict(w, doc_index=di, n_document_sentences=len(doc['text'])))
    return result


def class_weights(documents, labels, cap=5.0):
    counts = Counter(label for d in documents if d['split'] == 'train' for label in d['labels'])
    maximum = max(counts.values())
    weights = [min(math.sqrt(maximum / counts[k]), cap) if counts[k] else 0.0 for k in labels]
    return torch.tensor(weights, dtype=torch.float32), [counts[k] for k in labels]


def collate_windows(documents, windows, label_to_id, cap):
    rows = []
    for w in windows:
        doc = documents[w['doc_index']]
        chunks, owners, weights = [], [], []
        for owner, ids in enumerate(doc['token_ids'][w['start']:w['end']]):
            for chunk in split_chunks(ids, cap):
                chunks.append(chunk)
                owners.append(owner)
                weights.append(max(len(chunk) - 2, 1))
        rows.append((chunks, owners, weights, doc, w))
    if not rows:
        raise ValueError('Cannot collate an empty batch.')
    batch_size = len(rows)
    max_chunks = max(len(r[0]) for r in rows)
    max_tokens = max(len(c) for r in rows for c in r[0])
    max_sentences = max(r[4]['end'] - r[4]['start'] for r in rows)
    batch = {
        'ids': torch.full((batch_size, max_chunks, max_tokens), PAD_ID, dtype=torch.long),
        'lengths': torch.zeros(batch_size, max_chunks, dtype=torch.long),
        'owners': torch.zeros(batch_size, max_chunks, dtype=torch.long),
        'chunk_weights': torch.zeros(batch_size, max_chunks, dtype=torch.float32),
        'sentence_mask': torch.zeros(batch_size, max_sentences, dtype=torch.bool),
        'owned_mask': torch.zeros(batch_size, max_sentences, dtype=torch.bool),
        'targets': torch.full((batch_size, max_sentences), -100, dtype=torch.long),
        'use_start': torch.zeros(batch_size, dtype=torch.bool),
        'use_end': torch.zeros(batch_size, dtype=torch.bool),
    }
    for b, (chunks, owners, weights, doc, w) in enumerate(rows):
        n = w['end'] - w['start']
        batch['sentence_mask'][b, :n] = True
        lo, hi = w['owned_start'] - w['start'], w['owned_end'] - w['start']
        batch['owned_mask'][b, lo:hi] = True
        batch['use_start'][b] = w['owned_start'] == 0
        batch['use_end'][b] = w['owned_end'] == len(doc['text'])
        if doc['labels'] is not None:
            batch['targets'][b, :n] = torch.tensor([label_to_id[k] for k in doc['labels'][w['start']:w['end']]])
        for j, chunk in enumerate(chunks):
            batch['ids'][b, j, :len(chunk)] = torch.tensor(chunk)
            batch['lengths'][b, j] = len(chunk)
            batch['owners'][b, j] = owners[j]
            batch['chunk_weights'][b, j] = weights[j]
    return batch


def supervision_denominators(batch, weights):
    targets = batch['targets'][batch['owned_mask']]
    if (targets < 0).any():
        raise ValueError('An owned training position lacks a label.')
    return int(len(targets)), float(weights.cpu()[targets].sum())


def load_own_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    except RuntimeError as error:
        if 'mmap' not in str(error):
            raise
        return torch.load(path, map_location='cpu', weights_only=False)


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_data.py


In [7]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_shared.py
"""Established RR gates, unconstrained CRF, and trainable-state helpers."""
import torch
from torch import nn
import torch.nn.functional as F


class SentenceBiGRU(nn.Module):
    """One manually gated GRU per direction; reset-after convention."""
    def __init__(self, input_size, hidden):
        super().__init__()
        self.hidden = hidden
        self.x_layers = nn.ModuleList([nn.Linear(input_size, 3 * hidden) for _ in range(2)])
        self.h_layers = nn.ModuleList([nn.Linear(hidden, 3 * hidden) for _ in range(2)])

    def forward(self, x):
        outputs = []
        for direction in range(2):
            hidden = x.new_zeros(self.hidden)
            states = [None] * len(x)
            order = range(len(x)) if direction == 0 else range(len(x) - 1, -1, -1)
            for i in order:
                xr, xz, xn = self.x_layers[direction](x[i]).chunk(3)
                hr, hz, hn = self.h_layers[direction](hidden).chunk(3)
                reset, update = torch.sigmoid(xr + hr), torch.sigmoid(xz + hz)
                candidate = torch.tanh(xn + reset * hn)
                hidden = (1 - update) * candidate + update * hidden
                states[i] = hidden
            outputs.append(torch.stack(states))
        return torch.cat(outputs, dim=-1)


class LinearChainCRF(nn.Module):
    def __init__(self, n_labels):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(n_labels, n_labels))
        self.start = nn.Parameter(torch.zeros(n_labels))
        self.end = nn.Parameter(torch.zeros(n_labels))

    def log_partition(self, emissions, use_start=True, use_end=True):
        if len(emissions) == 0:
            return emissions.sum() * 0
        alpha = emissions[0] + (self.start if use_start else 0)
        for emission in emissions[1:]:
            alpha = emission + torch.logsumexp(alpha[:, None] + self.transitions, dim=0)
        return torch.logsumexp(alpha + (self.end if use_end else 0), dim=0)

    def gold_score(self, emissions, labels, use_start=True, use_end=True):
        if len(labels) == 0:
            return emissions.sum() * 0
        score = emissions.gather(1, labels[:, None]).sum()
        if len(labels) > 1:
            score = score + self.transitions[labels[:-1], labels[1:]].sum()
        if use_start:
            score = score + self.start[labels[0]]
        if use_end:
            score = score + self.end[labels[-1]]
        return score

    def nll(self, emissions, labels, use_start=True, use_end=True):
        return self.log_partition(emissions, use_start, use_end) - self.gold_score(emissions, labels, use_start, use_end)

    @torch.no_grad()
    def decode(self, emissions, use_start=True, use_end=True):
        if len(emissions) == 0:
            return []
        delta = emissions[0] + (self.start if use_start else 0)
        back = []
        for emission in emissions[1:]:
            scores, choices = (delta[:, None] + self.transitions).max(dim=0)
            delta = scores + emission
            back.append(choices)
        last = int((delta + (self.end if use_end else 0)).argmax())
        path = [last]
        for choices in reversed(back):
            last = int(choices[last])
            path.append(last)
        return list(reversed(path))


def bare(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in bare(model).named_parameters() if p.requires_grad}


def load_trainable_state(model, state):
    expected = {name: p for name, p in bare(model).named_parameters() if p.requires_grad}
    if set(expected) != set(state):
        raise ValueError('Checkpoint trainable parameter names differ.')
    with torch.no_grad():
        for name, p in expected.items():
            if tuple(p.shape) != tuple(state[name].shape):
                raise ValueError('Checkpoint parameter shape differs: ' + name)
            p.copy_(state[name].to(device=p.device, dtype=p.dtype))


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_shared.py


In [8]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_check_utils.py
import itertools
import torch
import torch.nn.functional as F
from rr_shared import LinearChainCRF


def crf_self_test():
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(19)
        crf = LinearChainCRF(3).double()
        with torch.no_grad():
            for p in crf.parameters():
                p.normal_(0, 0.2)
        emissions = torch.randn(4, 3, dtype=torch.float64, requires_grad=True)
        for use_start, use_end in ((True, True), (False, False), (True, False), (False, True)):
            paths = [torch.tensor(p) for p in itertools.product(range(3), repeat=4)]
            scores = torch.stack([crf.gold_score(emissions, p, use_start, use_end) for p in paths])
            torch.testing.assert_close(crf.log_partition(emissions, use_start, use_end), torch.logsumexp(scores, 0))
            expected = paths[int(scores.argmax())].tolist()
            assert crf.decode(emissions, use_start, use_end) == expected
            crf.nll(emissions, paths[0], use_start, use_end).backward(retain_graph=True)
            assert torch.isfinite(emissions.grad).all()
        zero = LinearChainCRF(3)
        one = torch.tensor([[0.2, -0.1, 0.8]])
        torch.testing.assert_close(zero.nll(one, torch.tensor([2])), F.cross_entropy(one, torch.tensor([2])))
        assert zero.decode(one) == [2]
    return {'status': 'passed', 'checks': ['partition vs exhaustive enumeration', 'Viterbi vs exhaustive search',
                                          'all four start/end boundary combinations', 'finite gradients', 'length-one CE identity']}


def learning_windows(documents, width=8):
    candidates = []
    for di, doc in enumerate(documents):
        if doc['split'] != 'train' or len(doc['text']) < width:
            continue
        for start in range(0, len(doc['text']) - width + 1, max(width // 2, 1)):
            labels = doc['labels'][start:start + width]
            changes = sum(a != b for a, b in zip(labels, labels[1:]))
            score = (len(set(labels)), changes)
            candidates.append((score, {'doc_index': di, 'start': start, 'end': start + width,
                'owned_start': start, 'owned_end': start + width, 'n_document_sentences': len(doc['text'])}))
    candidates.sort(key=lambda pair: pair[0], reverse=True)
    selected = []
    for _, w in candidates:
        if not selected or w['doc_index'] != selected[0]['doc_index']:
            selected.append(w)
        if len(selected) == 2:
            break
    if not selected:
        raise ValueError('No training windows available for the learning check.')
    if len(selected) == 1:
        selected.append(dict(selected[0]))
    return selected


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_check_utils.py


In [9]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_efficient_model.py
"""Efficient Phase 1 RR teacher and compact student; no training engine."""
import copy
from dataclasses import replace
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from architecture_models import EfficientLM, Attention
from rr_data import collate_windows
from rr_shared import SentenceBiGRU, LinearChainCRF


def valid_keys(lengths, width):
    return torch.arange(width, device=lengths.device)[None, :] < lengths[:, None]


def configure_backbone_dropout(lm, probability):
    """Change training dropout without changing or reinitializing pretrained weights."""
    probability = float(probability)
    if not 0 <= probability < 1:
        raise ValueError('Backbone dropout must be in [0, 1).')
    # cfg is copied, so the original Phase 1 configuration remains an identity record.
    lm.cfg = replace(lm.cfg, dropout=probability)
    for module in lm.modules():
        if isinstance(module, Attention):
            # SDPA reads this float directly; changing nn.Dropout alone misses it.
            module.dropout = probability
        elif isinstance(module, nn.Dropout):
            module.p = probability
    return probability


def dropout_audit(model):
    """Check the effective attention, residual, FFN and RR-head dropout settings."""
    base = model.module if isinstance(model, nn.DataParallel) else model
    expected = float(base.settings.get('backbone_dropout', base.lm.cfg.dropout))
    attention = [float(m.dropout) for m in base.lm.modules() if isinstance(m, Attention)]
    modules = [float(m.p) for m in base.lm.modules() if isinstance(m, nn.Dropout)]
    layers = len(base.lm.blocks)
    if (len(attention) != layers or len(modules) != 2 * layers or
            any(p != expected for p in attention + modules) or
            float(base.lm.cfg.dropout) != expected):
        raise ValueError('Effective backbone dropout does not match the recorded configuration.')
    head = float(base.dropout.p)
    if head != float(base.settings['head_dropout']):
        raise ValueError('Effective RR-head dropout differs from the recorded configuration.')
    return {'status':'passed', 'backbone_dropout':expected, 'head_dropout':head,
            'attention_probability_modules':len(attention),
            'attention_residual_and_ffn_dropout_modules':len(modules),
            'backbone_configuration_dropout':float(base.lm.cfg.dropout),
            'original_phase1_parameter_values_preserved_by_dropout_change':True}


def compact_lm(original, layers=(0, 7, 15)):
    if original.cfg.architecture != 'efficient' or len(set(layers)) != len(layers):
        raise ValueError('Use distinct blocks from the efficient Phase 1 model.')
    if not layers or min(layers) < 0 or max(layers) >= len(original.blocks):
        raise ValueError('Student block selection is outside the Phase 1 stack.')
    student = EfficientLM(replace(original.cfg, n_layers=len(layers)))
    student.tok_emb.load_state_dict(original.tok_emb.state_dict(), strict=True)
    student.final_norm.load_state_dict(original.final_norm.state_dict(), strict=True)
    for new, old in enumerate(layers):
        student.blocks[new].load_state_dict(original.blocks[old].state_dict(), strict=True)
    assert student.head.weight is student.tok_emb.weight
    assert student.tok_emb.weight.data_ptr() != original.tok_emb.weight.data_ptr()
    return student


def efficient_states(lm, ids, lengths, use_checkpoint=False):
    if ids.ndim != 2 or lengths.shape != (len(ids),):
        raise ValueError('Expected chunk IDs [batch,tokens] and lengths [batch].')
    if ids.shape[1] > lm.cfg.max_seq_len or not bool(((lengths >= 2) & (lengths <= ids.shape[1])).all()):
        raise ValueError('Chunk lengths exceed the original context budget.')
    allowed = valid_keys(lengths, ids.shape[1])
    x = lm.embed(ids.masked_fill(~allowed, 0))
    for block in lm.blocks:
        x = checkpoint(block, x, lengths, use_reentrant=False, preserve_rng_state=True) if use_checkpoint else block(x, lengths)
    return lm.final_norm(x)


def pool_content(states, lengths, mode='mean_last'):
    # Exclude BOS, EOS and padding; an empty sentence falls back to BOS.
    positions = torch.arange(states.shape[1], device=states.device)[None, :]
    keep = (positions > 0) & (positions < lengths[:, None] - 1)
    count = keep.sum(1)
    mean = (states.float() * keep[:, :, None]).sum(1) / count.clamp_min(1)[:, None]
    mean = torch.where((count > 0)[:, None], mean, states[:, 0].float())
    last_index = torch.where(count > 0, lengths - 2, torch.zeros_like(lengths))
    last = states[torch.arange(len(states), device=states.device), last_index].float()
    if mode == 'mean':
        return mean
    if mode == 'mean_last':
        return torch.cat([mean, last], -1)
    raise ValueError('Unknown content pooling mode.')


def rr_batch(documents, windows, labels, cap):
    result = collate_windows(documents, windows, {v: i for i, v in enumerate(labels)}, cap)
    features = torch.zeros((*result['sentence_mask'].shape, 4), dtype=torch.float32)
    for b, w in enumerate(windows):
        doc = documents[w['doc_index']]
        for local, absolute in enumerate(range(w['start'], w['end'])):
            relative = absolute / max(len(doc['text']) - 1, 1)
            features[b, local] = torch.tensor([relative, 1 - relative,
                min(len(doc['token_ids'][absolute]), 4096) / 4096,
                float(doc['domain'] == 'IT')])
    result['features'] = features
    return result


class EfficientRR(nn.Module):
    def __init__(self, lm, n_labels, weights, settings, adaptation='full'):
        super().__init__()
        if lm.cfg.architecture != 'efficient':
            raise ValueError('Use the efficient Phase 1 architecture.')
        self.lm, self.settings, self.n_labels = lm, dict(settings), n_labels
        self.backbone_dropout = configure_backbone_dropout(lm,
            settings.get('backbone_dropout', lm.cfg.dropout))
        width = lm.cfg.d_model * (2 if settings['pooling'] == 'mean_last' else 1)
        self.projection = nn.Linear(width + 4, settings['projection_dim'])
        self.context = SentenceBiGRU(settings['projection_dim'], settings['context_hidden'])
        self.dropout = nn.Dropout(settings['head_dropout'])
        self.classifier = nn.Linear(2 * settings['context_hidden'], n_labels)
        self.crf = LinearChainCRF(n_labels)
        self.register_buffer('class_weights', weights.detach().clone().float(), persistent=False)
        self.adaptation = adaptation
        for p in self.parameters():
            p.requires_grad_(True)
        if adaptation == 'upper4':
            for p in lm.parameters():
                p.requires_grad_(False)
            for block in lm.blocks[-4:]:
                for p in block.parameters():
                    p.requires_grad_(True)
            for p in lm.final_norm.parameters():
                p.requires_grad_(True)
        elif adaptation != 'full':
            raise ValueError('Choose full or upper4 fine-tuning.')

    def encode_chunks(self, ids, lengths):
        states = efficient_states(self.lm, ids, lengths,
            self.training and self.settings['gradient_checkpointing'])
        return pool_content(states, lengths, self.settings['pooling'])

    def one_window(self, ids, lengths, owners, weights, features, n):
        sizes = lengths.detach().cpu().tolist()
        count = sum(v > 0 for v in sizes)
        if count == 0:
            raise ValueError('Empty RR window.')
        chunks = []
        micro = self.settings['chunk_microbatch']
        for start in range(0, count, micro):
            stop = min(start + micro, count)
            chunks.append(self.encode_chunks(ids[start:stop, :max(sizes[start:stop])], lengths[start:stop]))
        chunks = torch.cat(chunks).float()
        w = weights[:count].float()
        pooled = chunks.new_zeros((n, chunks.shape[-1])).index_add(0, owners[:count], chunks * w[:, None])
        denominator = chunks.new_zeros(n).index_add(0, owners[:count], w)
        pooled = torch.cat([pooled / denominator[:, None], features[:n].float()], -1)
        h = self.dropout(torch.tanh(self.projection(pooled)))
        return self.classifier(self.dropout(self.context(h)))

    def forward(self, ids, lengths, owners, chunk_weights, sentence_mask, owned_mask,
                targets, use_start, use_end, features, teacher_probs=None, mode='loss'):
        results = []
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=ids.is_cuda):
            for b in range(len(ids)):
                n = int(sentence_mask[b].sum())
                logits = self.one_window(ids[b], lengths[b], owners[b], chunk_weights[b], features[b], n)
                if mode == 'emissions':
                    results.append(F.pad(logits.float(), (0, 0, 0, sentence_mask.shape[1] - n)))
                    continue
                keep = owned_mask[b, :n]
                scores, gold = logits[keep].float(), targets[b, :n][keep]
                if not len(gold) or bool((gold < 0).any()):
                    raise ValueError('An owned training sentence has no gold label.')
                with torch.autocast(device_type='cuda', enabled=False):
                    crf = self.crf.nll(scores, gold, bool(use_start[b]), bool(use_end[b]))
                    ce = F.cross_entropy(scores, gold, weight=self.class_weights, reduction='sum')
                    kd = scores.sum() * 0
                    if teacher_probs is not None:
                        q = teacher_probs[b, :n][keep].detach().float()
                        temperature = self.settings['temperature']
                        kd = F.kl_div(F.log_softmax(scores / temperature, -1), q,
                                      reduction='sum') * temperature ** 2
                device_id = ids.device.index if ids.is_cuda else -1
                results.append(torch.stack([crf, ce, kd, crf.new_tensor(float(device_id))]))
        return torch.stack(results)


def crf_marginals(crf, emissions, temperature=1.0, use_start=True, use_end=True):
    """Exact node marginals with ALL CRF potentials divided by temperature."""
    if temperature <= 0 or emissions.ndim != 2 or len(emissions) == 0:
        raise ValueError('Expected positive temperature and nonempty [sentences,labels] scores.')
    dtype = torch.float64 if emissions.dtype == torch.float64 else torch.float32
    e = emissions.to(dtype) / temperature
    a = crf.transitions.to(dtype) / temperature
    start = crf.start.to(dtype) / temperature if use_start else torch.zeros_like(e[0])
    end = crf.end.to(dtype) / temperature if use_end else torch.zeros_like(e[0])
    forward = [e[0] + start]
    for row in e[1:]:
        forward.append(row + torch.logsumexp(forward[-1][:, None] + a, 0))
    backward = [end]
    for i in range(len(e) - 2, -1, -1):
        backward.append(torch.logsumexp(a + e[i + 1][None, :] + backward[-1][None, :], 1))
    backward.reverse()
    return torch.softmax(torch.stack(forward) + torch.stack(backward), -1)


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_efficient_model.py


In [10]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_efficient_probe.py
"""Disposable preflight probes. Never writes trained weights or a real KD cache."""
import copy
import gc
import hashlib
import itertools
import json
import math
from pathlib import Path
import random
import statistics
import time
import traceback
import zipfile
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from rr_data import make_windows, split_chunks, supervision_denominators, write_json
from rr_shared import bare, LinearChainCRF
from rr_check_utils import crf_self_test, learning_windows
from rr_efficient_model import (compact_lm, EfficientRR, rr_batch, efficient_states,
                                pool_content, crf_marginals)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sync():
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            torch.cuda.synchronize(i)


def clean():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def feed(model, batch, mode='loss'):
    if isinstance(model, nn.DataParallel):
        return model(**batch, mode=mode)
    device = next(model.parameters()).device
    return model(**{k: v.to(device) for k, v in batch.items()}, mode=mode)


def objective(stats, sentences, weight_sum, settings, kd_weight=0.0):
    return (stats[:, 0].sum() / sentences
            + settings['aux_ce_coefficient'] * stats[:, 1].sum() / weight_sum
            + kd_weight * stats[:, 2].sum() / sentences)


def optimizer_for(model, backbone_lr, settings, multiplier=1.0):
    groups = {}
    for name, p in bare(model).named_parameters():
        if not p.requires_grad:
            continue
        kind = 'backbone' if name.startswith('lm.') else ('crf' if name.startswith('crf.') else 'head')
        lr = backbone_lr if kind == 'backbone' else settings[kind + '_lr']
        decay = settings['weight_decay'] if kind != 'crf' and p.ndim > 1 else 0.0
        groups.setdefault((kind, decay), {'params': [], 'lr': lr * multiplier,
            'weight_decay': decay, 'kind': kind})['params'].append(p)
    return torch.optim.AdamW(list(groups.values()), betas=(0.9, 0.999), eps=1e-8, foreach=False)


def gradient_record(model, collect_samples=False):
    groups = {'backbone': 0.0, 'head': 0.0, 'crf': 0.0}
    samples = []
    sample_names = []
    for name, p in bare(model).named_parameters():
        if not p.requires_grad:
            if p.grad is not None:
                raise AssertionError('Frozen parameter received a gradient: ' + name)
            continue
        if p.grad is None:
            if name in ('crf.start', 'crf.end'):
                if collect_samples:
                    samples.append(torch.zeros(min(16, p.numel()), dtype=torch.float32))
                    sample_names.append(name)
                continue
            raise AssertionError('Missing trainable gradient: ' + name)
        if not bool(torch.isfinite(p.grad).all()):
            raise AssertionError('Nonfinite gradient: ' + name)
        group = 'backbone' if name.startswith('lm.') else ('crf' if name.startswith('crf.') else 'head')
        groups[group] += float(p.grad.detach().float().square().sum())
        if collect_samples:
            flat = p.grad.detach().flatten()
            indices = torch.linspace(0, len(flat) - 1, min(16, len(flat)), device=flat.device).long()
            samples.append(flat[indices].float().cpu())
            sample_names.append(name)
    result = {'squared_l2': groups}
    if collect_samples:
        result['sample'] = torch.cat(samples)
        result['sample_names'] = sample_names
    return result


def math_checks():
    report = {'crf': crf_self_test()}
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(23)
        for length in (1, 4):
            crf = LinearChainCRF(3).double()
            with torch.no_grad():
                for p in crf.parameters():
                    p.normal_(0, 0.3)
            e = torch.randn(length, 3, dtype=torch.float64)
            paths = [torch.tensor(v) for v in itertools.product(range(3), repeat=length)]
            for temperature in (1.0, 2.0):
                for use_start, use_end in ((True, True), (False, False), (True, False), (False, True)):
                    scores = torch.stack([crf.gold_score(e, p, use_start, use_end) for p in paths]) / temperature
                    probabilities = scores.softmax(0)
                    expected = torch.zeros_like(e)
                    for p, probability in zip(paths, probabilities):
                        expected[torch.arange(length), p] += probability
                    actual = crf_marginals(crf, e, temperature, use_start, use_end)
                    torch.testing.assert_close(actual, expected, atol=1e-10, rtol=1e-10)
        z = torch.randn(5, 3, dtype=torch.float64, requires_grad=True)
        q = (z.detach() / 2).softmax(-1)
        loss = 4 * F.kl_div((z / 2).log_softmax(-1), q, reduction='batchmean')
        loss.backward()
        assert abs(float(loss.detach())) < 1e-10 and float(z.grad.abs().max()) < 1e-10
        changed = (z.detach() + torch.tensor([0.5, -0.3, 0.1])).requires_grad_(True)
        positive = 4 * F.kl_div((changed / 2).log_softmax(-1), q, reduction='batchmean')
        positive.backward()
        assert float(positive.detach()) > 0 and float(changed.grad.abs().sum()) > 0
        states = torch.arange(2 * 6 * 4).reshape(2, 6, 4).float()
        lengths = torch.tensor([5, 2])
        pooled = pool_content(states, lengths)
        torch.testing.assert_close(pooled[0, :4], states[0, 1:4].mean(0))
        torch.testing.assert_close(pooled[0, 4:], states[0, 3])
        torch.testing.assert_close(pooled[1], torch.cat([states[1, 0], states[1, 0]]))
    report.update(status='passed', crf_marginals='exhaustive enumeration at T=1 and T=2; all boundaries',
                  kd='zero for matching distributions; positive loss and active gradient after perturbation',
                  pooling='mean, final content token, specials, padding and empty content checked')
    return report


def data_checks(documents, labels, settings):
    result = {}
    for split in ('train', 'dev', 'test'):
        windows = make_windows(documents, settings['window_sentences'], settings['overlap_sentences'], split)
        ownership = {i: np.zeros(len(d['text']), dtype=np.int32)
                     for i, d in enumerate(documents) if d['split'] == split}
        for w in windows:
            ownership[w['doc_index']][w['owned_start']:w['owned_end']] += 1
        if not all(bool((v == 1).all()) for v in ownership.values()):
            raise AssertionError('Sentence window ownership is not exactly once.')
        result[split] = {'documents': len(ownership), 'sentences': sum(map(len, ownership.values())), 'windows': len(windows)}
    for doc in documents:
        for tokens in doc['token_ids']:
            chunks = split_chunks(tokens, settings['chunk_tokens'])
            assert [v for chunk in chunks for v in chunk[1:-1]] == tokens
            assert all(2 <= len(c) <= settings['chunk_tokens'] for c in chunks)
    # Real overlap window: non-owned positions may NEVER enter gold or KD loss.
    overlapping = next((w for w in make_windows(documents, settings['window_sentences'], settings['overlap_sentences'], 'train')
        if w['owned_start'] > w['start'] or w['owned_end'] < w['end']), None)
    if overlapping is None:
        raise AssertionError('Expected at least one long training document.')
    result.update(status='passed', exact_token_reassembly=True, overlap_example=overlapping,
                  official_labels=labels)
    return result


@torch.no_grad()
def original_parity(lm):
    device = next(lm.parameters()).device
    ids = torch.tensor([[2, 10, 21, 32, 43, 54, 65, 3, 0, 0],
                        [2, 11, 22, 33, 3, 0, 0, 0, 0, 0]], device=device)
    lengths = torch.tensor([8, 5], device=device)
    lm.eval()
    actual = efficient_states(lm, ids, lengths)
    original = lm(ids, lengths)
    expected = lm.head(actual)
    valid = torch.arange(ids.shape[1], device=device)[None, :] < lengths[:, None]
    torch.testing.assert_close(original[valid], expected[valid], rtol=2e-4, atol=2e-4)
    changed = ids.clone()
    changed[~valid] = 99
    padding = efficient_states(lm, changed, lengths)
    torch.testing.assert_close(actual[valid], padding[valid], rtol=2e-4, atol=2e-4)
    changed = ids.clone()
    changed[:, 4:] = (changed[:, 4:] + 37) % lm.cfg.vocab_size
    future = efficient_states(lm, changed, lengths)
    torch.testing.assert_close(actual[:, :4], future[:, :4], rtol=2e-4, atol=2e-4)
    return {'status': 'passed', 'original_forward_parity': True, 'padding_invariance': True,
            'future_token_causality': True, 'original_gqa_and_local_attention_retained': True}


def provenance(original, student, selected):
    pairs = [('embedding', original.tok_emb, student.tok_emb),
             ('final_norm', original.final_norm, student.final_norm)]
    pairs += [(f'block_{i}_from_{j}', original.blocks[j], student.blocks[i]) for i, j in enumerate(selected)]
    for name, source, destination in pairs:
        for k, value in source.state_dict().items():
            assert torch.equal(value, destination.state_dict()[k]), (name, k)
        for a, b in zip(source.parameters(), destination.parameters()):
            assert a.data_ptr() != b.data_ptr(), name
    return {'status': 'passed', 'block_mapping': dict(enumerate(selected)),
            'copied_values_exact': True, 'independent_parameter_storage': True,
            'original_backbone_parameters': sum(p.numel() for p in original.parameters()),
            'student_backbone_parameters': sum(p.numel() for p in student.parameters())}


def parallel_check(model, documents, labels, weights, settings):
    batch = rr_batch(documents, learning_windows(documents), labels, settings['chunk_tokens'])
    count, weight_sum = supervision_denominators(batch, weights)
    model.eval()
    model.zero_grad(set_to_none=True)
    stats = feed(model, batch)
    loss = objective(stats, count, weight_sum, settings)
    (128 * loss).backward()
    record = gradient_record(model, collect_samples=True)
    devices = sorted(set(int(v) for v in stats[:, 3].detach().cpu()))
    if isinstance(model, nn.DataParallel):
        if devices != model.device_ids:
            raise AssertionError('Both configured GPUs must participate in the training batch.')
        model.zero_grad(set_to_none=True)
        serial = feed(bare(model), batch)
        reference = objective(serial, count, weight_sum, settings)
        (128 * reference).backward()
        second = gradient_record(model, collect_samples=True)
        if record['sample_names'] != second['sample_names']:
            raise AssertionError('Parallel/serial gradient parameter names differ.')
        relative = float((record['sample'] - second['sample']).norm() / second['sample'].norm().clamp_min(1e-15))
        difference = abs(float(loss.detach()) - float(reference.detach()))
        if relative > 0.04 or difference > 0.015:
            raise AssertionError(f'Parallel normalization mismatch: loss={difference}, sampled gradient={relative}.')
    else:
        relative, difference = 0.0, 0.0
    model.zero_grad(set_to_none=True)
    return {'status': 'passed', 'devices_seen': devices, 'loss_difference': difference,
            'sampled_gradient_relative_l2': relative,
            'gradient_groups_squared_l2': record['squared_l2']}


@torch.no_grad()
def mini_metrics(model, batch):
    model.eval()
    logits = feed(model, batch, 'emissions').cpu()[batch['owned_mask']]
    gold = batch['targets'][batch['owned_mask']]
    return {'ce': float(F.cross_entropy(logits, gold)),
            'accuracy': float((logits.argmax(-1) == gold).float().mean())}


def one_update(model, batches, weights, settings, optimizer, scaler, kd_weight=0.0):
    denominators = [supervision_denominators(b, weights) for b in batches]
    count, weight_sum = sum(x[0] for x in denominators), sum(x[1] for x in denominators)
    optimizer.zero_grad(set_to_none=True)
    model.train()
    sync()
    started = time.perf_counter()
    devices = set()
    total_loss = 0.0
    for batch in batches:
        stats = feed(model, batch)
        loss = objective(stats, count, weight_sum, settings, kd_weight)
        if not bool(torch.isfinite(loss)):
            raise AssertionError('Nonfinite forward objective.')
        scaler.scale(loss).backward()
        devices.update(int(v) for v in stats[:, 3].detach().cpu())
        total_loss += float(loss.detach())
    scaler.unscale_(optimizer)
    norm = nn.utils.clip_grad_norm_([p for p in bare(model).parameters() if p.requires_grad], settings['gradient_clip'])
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    skipped = scaler.get_scale() < old_scale
    if not skipped and not bool(torch.isfinite(norm)):
        raise AssertionError('Nonfinite unscaled gradient norm.')
    sync()
    return {'seconds': time.perf_counter() - started, 'loss': total_loss,
            'amp_skipped': skipped, 'gradient_norm': float(norm) if torch.isfinite(norm) else None,
            'devices_seen': sorted(devices)}


def learning_check(model, documents, labels, weights, settings, steps):
    batch = rr_batch(documents, learning_windows(documents), labels, settings['chunk_tokens'])
    before = mini_metrics(model, batch)
    optimizer = optimizer_for(model, settings['student_lr'], settings, multiplier=5.0)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available(), init_scale=128)
    curve = []
    for step in range(steps):
        update = one_update(model, [batch], weights, settings, optimizer, scaler)
        curve.append(dict(step=step + 1, **update))
        if step == 0 or (step + 1) % 10 == 0:
            print('Disposable student learning:', step + 1, '/', steps, '| loss', round(update['loss'], 4), flush=True)
    after = mini_metrics(model, batch)
    passed = after['ce'] < before['ce'] * 0.85 and (after['accuracy'] >= before['accuracy'] + 0.10 or after['accuracy'] >= 0.90)
    skipped = sum(v['amp_skipped'] for v in curve)
    if skipped > max(2, steps // 10):
        passed = False
    optimizer.zero_grad(set_to_none=True)
    del optimizer, scaler
    return {'status': 'passed' if passed else 'failed', 'before': before, 'after': after,
            'temporary_lr_multiplier': 5.0, 'steps': steps, 'amp_skipped': skipped, 'curve': curve}


def window_cost(documents, w, cap):
    values = documents[w['doc_index']]['token_ids'][w['start']:w['end']]
    return sum(max(1, math.ceil(len(v) / (cap - 2))) for v in values), sum(len(v) for v in values)


@torch.no_grad()
def reconstruct(model, documents, labels, settings, split='dev', selected_indices=None):
    windows = make_windows(documents, settings['window_sentences'], settings['overlap_sentences'], split)
    indices = [i for i, d in enumerate(documents) if d['split'] == split and (selected_indices is None or i in selected_indices)]
    windows = [w for w in windows if w['doc_index'] in indices]
    scores = {i: torch.empty(len(documents[i]['text']), len(labels)) for i in indices}
    counts = {i: np.zeros(len(documents[i]['text']), dtype=np.int32) for i in indices}
    model.eval()
    sync()
    started = time.perf_counter()
    for start in range(0, len(windows), settings['global_window_batch']):
        group = windows[start:start + settings['global_window_batch']]
        batch = rr_batch(documents, group, labels, settings['chunk_tokens'])
        logits = feed(model, batch, 'emissions').cpu()
        for b, w in enumerate(group):
            lo, hi = w['owned_start'] - w['start'], w['owned_end'] - w['start']
            scores[w['doc_index']][w['owned_start']:w['owned_end']] = logits[b, lo:hi]
            counts[w['doc_index']][w['owned_start']:w['owned_end']] += 1
    sync()
    seconds = time.perf_counter() - started
    for i in indices:
        if not bool((counts[i] == 1).all()) or not bool(torch.isfinite(scores[i]).all()):
            raise AssertionError('Inference reconstruction lost, duplicated or corrupted a sentence.')
    cpu_crf = copy.deepcopy(bare(model).crf).cpu()
    for i in indices:
        path = cpu_crf.decode(scores[i])
        assert len(path) == len(documents[i]['text']) and all(0 <= v < len(labels) for v in path)
    return scores, cpu_crf, {'seconds': seconds, 'documents': len(indices),
                           'sentences': sum(len(documents[i]['text']) for i in indices), 'windows': len(windows)}


def text_hash(doc):
    return hashlib.sha256(json.dumps(doc['text'], ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()


def validate_cache(metadata, arrays, documents, labels, expected_identity, temperature):
    if metadata['purpose'] != 'preflight_wiring_only':
        raise ValueError('This validator is only for the disposable preflight cache.')
    if metadata['identity'] != expected_identity or metadata['labels'] != labels or metadata['temperature'] != temperature:
        raise ValueError('Cache identity, label order or temperature differs.')
    if len({r['document_id'] for r in metadata['documents']}) != len(metadata['documents']):
        raise ValueError('Duplicate cache document ID.')
    by_id = {d['id']: d for d in documents if d['split'] == 'train'}
    for r in metadata['documents']:
        d = by_id.get(r['document_id'])
        if d is None or r['text_sha256'] != text_hash(d) or r['split'] != 'train':
            raise ValueError('Cache document/split/sentence identity differs.')
        q, index = arrays[r['key']], arrays[r['key'] + '_indices']
        if q.shape != (len(d['text']), len(labels)) or not np.array_equal(index, np.arange(len(d['text']))):
            raise ValueError('Cache sentence indices or probability shape differs.')
        if not np.isfinite(q).all() or (q < 0).any() or not np.allclose(q.sum(-1), 1, atol=2e-5):
            raise ValueError('Cache probabilities are invalid.')
    return True


def cache_probe(model, documents, labels, settings, output_dir, identity):
    selected = [min((i for i, d in enumerate(documents) if d['split'] == 'train' and d['domain'] == domain and len(d['text']) >= 2),
                    key=lambda i: len(documents[i]['text'])) for domain in ('CL', 'IT')]
    scores, crf, timing = reconstruct(model, documents, labels, settings, 'train', selected)
    metadata = {'purpose': 'preflight_wiring_only', 'warning': 'Temporary RR probe; NOT a trained teacher. Never use for full KD.',
                'identity': identity, 'temperature': settings['temperature'], 'labels': labels, 'documents': []}
    arrays = {}
    for j, i in enumerate(selected):
        key = 'q_' + str(j)
        q = crf_marginals(crf, scores[i], settings['temperature']).detach().numpy().astype(np.float32)
        arrays[key], arrays[key + '_indices'] = q, np.arange(len(q))
        metadata['documents'].append({'document_id': documents[i]['id'], 'split': 'train',
                                      'text_sha256': text_hash(documents[i]), 'key': key})
    path = Path(output_dir) / 'temporary_cache_wiring_test.npz'
    np.savez_compressed(path, header=np.array(json.dumps(metadata)), **arrays)
    with np.load(path, allow_pickle=False) as saved:
        restored = json.loads(str(saved['header'].item()))
        loaded = {k: saved[k].copy() for k in saved.files if k != 'header'}
    validate_cache(restored, loaded, documents, labels, identity, settings['temperature'])
    for k, v in arrays.items():
        assert np.array_equal(v, loaded[k])
    for change in ('label_order', 'document_id', 'sentence_index', 'probability'):
        bad, altered = copy.deepcopy(restored), dict(loaded)
        if change == 'label_order':
            bad['labels'] = list(reversed(labels))
        elif change == 'document_id':
            bad['documents'][0]['document_id'] = 'incorrect-document'
        elif change == 'sentence_index':
            key = bad['documents'][0]['key'] + '_indices'
            altered[key] = altered[key] + 1
        else:
            key = bad['documents'][0]['key']
            altered[key] = altered[key].copy()
            altered[key][0, 0] = float('nan')
        try:
            validate_cache(bad, altered, documents, labels, identity, settings['temperature'])
        except ValueError:
            pass
        else:
            raise AssertionError('Cache validator accepted corruption: ' + change)
    return {'status': 'passed', 'purpose': metadata['purpose'], 'inference': timing,
            'documents': metadata['documents'], 'path': str(path),
            'corrupt_label_order_document_index_and_probability_rejected': True}, (metadata, arrays)


def kd_gradient_check(model, documents, labels, weights, settings, cache):
    metadata, arrays = cache
    by_id = {d['id']: i for i, d in enumerate(documents)}
    windows = []
    qs = []
    for row in metadata['documents']:
        i = by_id[row['document_id']]
        start = max(0, len(documents[i]['text']) // 2 - 4)
        end = min(start + 8, len(documents[i]['text']))
        # Interior non-owned sentence checks that padding/overlap targets are excluded.
        windows.append(dict(doc_index=i, start=start, end=end, owned_start=start + 1,
            owned_end=end, n_document_sentences=len(documents[i]['text'])))
        qs.append(arrays[row['key']][start:end])
    batch = rr_batch(documents, windows, labels, settings['chunk_tokens'])
    q = torch.full((*batch['sentence_mask'].shape, len(labels)), float('nan'))
    for b, values in enumerate(qs):
        q[b, :len(values)] = torch.from_numpy(values)
    q[~batch['owned_mask']] = float('nan')
    batch['teacher_probs'] = q
    count, weight_sum = supervision_denominators(batch, weights)
    model.eval()
    model.zero_grad(set_to_none=True)
    stats = feed(model, batch)
    loss = objective(stats, count, weight_sum, settings, settings['kd_weight'])
    if not bool(torch.isfinite(loss)):
        raise AssertionError('KD padding/overlap exclusion failed.')
    (128 * loss).backward()
    gradients = gradient_record(model)
    kd = float(stats[:, 2].sum().detach()) / count
    if kd < -1e-5 or q.requires_grad or gradients['squared_l2']['backbone'] <= 0:
        raise AssertionError('Invalid KD loss or gradient path.')
    model.zero_grad(set_to_none=True)
    return {'status': 'passed', 'kd_per_owned_sentence': kd,
            'teacher_detached': True, 'non_owned_nan_targets_ignored': True, 'gradients': gradients}


def diagnostic_zip(output_dir):
    output_dir = Path(output_dir)
    archive_path = output_dir.parent.parent / (output_dir.parent.name + '_diagnostics.zip')
    archive_path.parent.mkdir(exist_ok=True, parents=True)
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(output_dir.rglob('*')):
            if p.is_file() and p.suffix in ('.json', '.csv', '.txt', '.py', '.npz'):
                z.write(p, str(Path(output_dir.parent.name) / output_dir.name / p.relative_to(output_dir)))
    return archive_path


def run_case(factory, name, documents, labels, weights, settings, output_dir, cache=None):
    report = {'name': name, 'status': 'running'}
    model = base = optimizer = scaler = None
    produced_cache = None
    try:
        seed_all(settings['seed'])
        base = factory()
        from rr_efficient_model import dropout_audit
        report['effective_dropout'] = dropout_audit(base)
        report['parameters'] = {'total': sum(p.numel() for p in base.parameters()),
            'trainable': sum(p.numel() for p in base.parameters() if p.requires_grad),
            'trainable_backbone': sum(p.numel() for p in base.lm.parameters() if p.requires_grad)}
        base = base.to('cuda:0')
        report['identity'] = original_parity(base.lm)
        model = nn.DataParallel(base, device_ids=[0, 1])
        report['parallel'] = parallel_check(model, documents, labels, weights, settings)
        if name == 'student':
            if cache is not None:
                report['kd'] = kd_gradient_check(model, documents, labels, weights, settings, cache)
            # This model is disposed after learning; resource timing uses a fresh copy below.
            report['learning'] = learning_check(model, documents, labels, weights, settings, settings['learning_steps'])
            if report['learning']['status'] != 'passed':
                raise AssertionError('Student tiny-set learning check failed; inspect curve before full training.')
            model.zero_grad(set_to_none=True)
            del model, base
            model = base = None
            clean()
            seed_all(settings['seed'])
            base = factory().to('cuda:0')
            model = nn.DataParallel(base, device_ids=[0, 1])
        optimizer = optimizer_for(model, settings['student_lr'] if name == 'student' else settings['teacher_lr'], settings)
        scaler = torch.amp.GradScaler('cuda', init_scale=128)
        windows = make_windows(documents, settings['window_sentences'], settings['overlap_sentences'], 'train')
        ordered = sorted(windows, key=lambda w: window_cost(documents, w, settings['chunk_tokens']), reverse=True)
        size, accumulation = settings['global_window_batch'], settings['accumulation_steps']
        heavy = [rr_batch(documents, ordered[j * size:(j + 1) * size], labels, settings['chunk_tokens'])
                 for j in range(accumulation)]
        report['stress_windows'] = [dict(w, document_id=documents[w['doc_index']]['id'],
            chunks=window_cost(documents, w, settings['chunk_tokens'])[0]) for w in ordered[:size * accumulation]]
        report['warmup'] = one_update(model, heavy, weights, settings, optimizer, scaler)
        for i in range(2):
            torch.cuda.reset_peak_memory_stats(i)
        report['heavy_updates'] = [one_update(model, heavy, weights, settings, optimizer, scaler) for _ in range(2)]
        rng = random.Random(settings['seed'])
        sampled = rng.sample(windows, min(len(windows), size * accumulation * 3))
        representative = []
        for j in range(3):
            chosen = sampled[j * size * accumulation:(j + 1) * size * accumulation]
            if len(chosen) != size * accumulation:
                break
            batches = [rr_batch(documents, chosen[k:k + size], labels, settings['chunk_tokens']) for k in range(0, len(chosen), size)]
            representative.append(one_update(model, batches, weights, settings, optimizer, scaler))
        report['representative_updates'] = representative
        if any(v['amp_skipped'] for v in report['heavy_updates'] + representative):
            raise AssertionError('AMP skipped a measured update; inspect numerical stability before training.')
        report['final_gradients'] = gradient_record(model)
        report['peak_allocated_gib'] = [torch.cuda.max_memory_allocated(i) / 2**30 for i in range(2)]
        report['peak_reserved_gib'] = [torch.cuda.max_memory_reserved(i) / 2**30 for i in range(2)]
        optimizer.zero_grad(set_to_none=True)
        del optimizer, scaler
        optimizer = scaler = None
        clean()
        _, _, dev = reconstruct(model, documents, labels, settings)
        report['dev_reconstruction'] = dev
        updates_per_epoch = math.ceil(math.ceil(len(windows) / size) / accumulation)
        update_seconds = statistics.mean(v['seconds'] for v in representative)
        dev_windows = make_windows(documents, settings['window_sentences'], settings['overlap_sentences'], 'dev')
        token_cost = lambda ws: sum(window_cost(documents, w, settings['chunk_tokens'])[1] for w in ws)
        train_eval_seconds = dev['seconds'] * token_cost(windows) / max(token_cost(dev_windows), 1)
        epoch_seconds = (updates_per_epoch * update_seconds + dev['seconds'] + train_eval_seconds / 3) * 1.20
        budget = settings['student_training_hours'] if name == 'student' else settings['teacher_training_hours']
        report['timing'] = {'updates_per_epoch': updates_per_epoch, 'mean_update_seconds': update_seconds,
            'estimated_epoch_minutes_with_20pct_margin': epoch_seconds / 60,
            'estimated_25_epoch_hours': epoch_seconds * 25 / 3600,
            'estimated_35_epoch_hours': epoch_seconds * 35 / 3600,
            'budget_hours': budget, 'epochs_fitting_budget_capped_35': min(35, int(budget * 3600 / epoch_seconds)),
            'includes': 'dev every epoch, full train inference estimated every third epoch, 20% allowance',
            'limitations': 'Only three sampled updates. Check first real epoch; I/O and CPU metric overhead vary.'}
        if name.startswith('teacher'):
            report['cache'], produced_cache = cache_probe(model, documents, labels, settings, output_dir, settings['cache_identity'])
        report['status'] = 'passed'
    except torch.cuda.OutOfMemoryError:
        report.update(status='oom', failure='CUDA out of memory', traceback=traceback.format_exc())
    except Exception as e:
        report.update(status='failed', failure=str(e), traceback=traceback.format_exc())
    finally:
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        if model is not None:
            model.zero_grad(set_to_none=True)
        model = base = optimizer = scaler = None
        clean()
        write_json(Path(output_dir) / (name + '_report.json'), report)
    return report, produced_cache


def run_preflight(original, documents, labels, weights, settings, output_dir):
    output_dir = Path(output_dir)
    report = {'status': 'running', 'settings': settings, 'cases': [],
              'temporary_updates_only': True, 'trained_teacher_quality_not_tested': True}
    saved_rng = {'python': random.getstate(), 'numpy': np.random.get_state(),
                 'torch': torch.get_rng_state(), 'cuda': torch.cuda.get_rng_state_all()}
    started = time.perf_counter()
    try:
        report['math'] = math_checks()
        report['data'] = data_checks(documents, labels, settings)
        compact = compact_lm(original, tuple(settings['student_layers']))
        report['student_provenance'] = provenance(original, compact, tuple(settings['student_layers']))
        del compact
        clean()
        cache = None
        for kind, adaptation in [('teacher_full', 'full'), ('teacher_upper4', 'upper4'), ('student', 'full')]:
            current = dict(settings)
            success = False
            for micro in dict.fromkeys((settings['chunk_microbatch'], 4, 2)):
                if micro > settings['chunk_microbatch']:
                    continue
                current['chunk_microbatch'] = micro
                print('\nStarting', kind, '| chunk microbatch', micro, flush=True)
                def factory():
                    lm = compact_lm(original, tuple(settings['student_layers'])) if kind == 'student' else copy.deepcopy(original)
                    return EfficientRR(lm, len(labels), weights, current, adaptation)
                case_dir = output_dir / (kind + '_micro' + str(micro))
                case_dir.mkdir(parents=True, exist_ok=True)
                case, produced = run_case(factory, kind, documents, labels, weights, current, case_dir, cache)
                case['chunk_microbatch'] = micro
                report['cases'].append(case)
                print(kind, '|', case['status'], flush=True)
                write_json(output_dir / 'preflight_report.json', report)
                if case['status'] == 'passed':
                    success = True
                    if cache is None and produced is not None:
                        cache = produced
                    break
                if case['status'] != 'oom':
                    break
            if kind == 'student' and not success:
                raise AssertionError('Compact student checks did not pass.')
        teachers = [c for c in report['cases'] if c['name'].startswith('teacher') and c['status'] == 'passed']
        students = [c for c in report['cases'] if c['name'] == 'student' and c['status'] == 'passed']
        if not teachers or not students or cache is None:
            raise AssertionError('Need a passing teacher resource/cache probe and a passing compact student.')
        student = students[-1]
        if student.get('kd', {}).get('status') != 'passed':
            raise AssertionError('Student KD gradient/cache alignment check did not pass.')
        fitting = [t for t in teachers if t['timing']['estimated_25_epoch_hours'] <= settings['teacher_training_hours']]
        chosen = (fitting or teachers)[0]
        report['recommendation'] = {'teacher_case': chosen['name'], 'teacher_microbatch': chosen['chunk_microbatch'],
            'student_microbatch': student['chunk_microbatch'], 'teacher_timing': chosen['timing'], 'student_timing': student['timing'],
            'full_training_not_started': True,
            'next': 'Review diagnostics before choosing teacher adaptation, epoch caps and starting real runs.'}
        report['status'] = 'passed'
    except Exception as e:
        report.update(status='failed', failure=str(e), traceback=traceback.format_exc())
    finally:
        random.setstate(saved_rng['python'])
        np.random.set_state(saved_rng['numpy'])
        torch.set_rng_state(saved_rng['torch'])
        torch.cuda.set_rng_state_all(saved_rng['cuda'])
        report['elapsed_minutes'] = (time.perf_counter() - started) / 60
        report['temporary_models_discarded'] = True
        report['real_teacher_cache_created'] = False
        write_json(output_dir / 'preflight_report.json', report)
    return report, diagnostic_zip(output_dir)


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_efficient_probe.py


In [11]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_efficient_engine.py
"""Efficient RR teacher/student: full-document evaluation, training, checkpoints and export."""
import copy
import gc
import hashlib
import json
import math
from pathlib import Path
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, precision_recall_fscore_support

from rr_data import collate_windows, supervision_denominators, write_json, make_windows
from rr_shared import bare, trainable_state, load_trainable_state, LinearChainCRF
from rr_efficient_probe import optimizer_for
from rr_efficient_model import rr_batch, crf_marginals


def assert_protocol_matches(saved, current):
    if saved == current:
        return
    differences = []
    def visit(old, new, path):
        if old == new:
            return
        if isinstance(old, dict) and isinstance(new, dict):
            for key in sorted(set(old) | set(new)):
                visit(old.get(key), new.get(key), path + '.' + key if path else key)
        else:
            differences.append(path + ': saved=' + repr(old)[:150] + ', current=' + repr(new)[:150])
    visit(saved, current, '')
    raise ValueError('Saved RR protocol differs. Restore its settings/source or use a new RUN_ID.\n' + '\n'.join(differences[:20]))


def rng_state():
    return {'python': random.getstate(), 'numpy': np.random.get_state(), 'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    for i, saved in enumerate(state['cuda'][:torch.cuda.device_count()]):
        torch.cuda.set_rng_state(saved.cpu(), i)




def scheduler_for(optimizer, total_updates, settings):
    warmup = max(1, math.ceil(total_updates * settings['warmup_fraction']))
    minimum = settings['minimum_lr_fraction']
    def factor(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, (step - warmup) / max(total_updates - warmup - 1, 1))
        return minimum + (1 - minimum) * 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def feed(model, batch, mode='loss'):
    if isinstance(model, nn.DataParallel):
        return model(**batch, mode=mode)
    device = next(model.parameters()).device
    return model(**{key: value.to(device) for key, value in batch.items()}, mode=mode)


def combined_loss(stats, sentence_count, weight_sum, ce_coefficient, kd_coefficient=0.0):
    if sentence_count <= 0 or weight_sum <= 0:
        raise ValueError('A training update requires positive supervision denominators.')
    return (stats[:, 0].sum() / sentence_count + ce_coefficient * stats[:, 1].sum() / weight_sum
            + kd_coefficient * stats[:, 2].sum() / sentence_count)


def cpu_crf(model):
    base = bare(model)
    crf = LinearChainCRF(base.n_labels)
    crf.load_state_dict({k: v.detach().float().cpu() for k, v in base.crf.state_dict().items()})
    return crf.eval()


@torch.no_grad()
def predict_emissions(model, documents, windows, labels, split, settings):
    model.eval()
    label_to_id = {name: i for i, name in enumerate(labels)}
    records = {i: np.zeros((len(doc['text']), len(labels)), dtype=np.float32)
               for i, doc in enumerate(documents) if doc['split'] == split}
    covered = {i: np.zeros(len(documents[i]['text']), dtype=bool) for i in records}
    started = time.monotonic()
    for start in range(0, len(windows), settings['global_window_batch']):
        part = windows[start:start + settings['global_window_batch']]
        batch = rr_batch(documents, part, labels, settings['chunk_tokens'])
        emissions = feed(model, batch, mode='emissions').float().cpu().numpy()
        for b, w in enumerate(part):
            di = w['doc_index']
            lo, hi = w['owned_start'], w['owned_end']
            if covered[di][lo:hi].any():
                raise ValueError('A sentence was assigned emissions more than once.')
            records[di][lo:hi] = emissions[b, lo - w['start']:hi - w['start']]
            covered[di][lo:hi] = True
    if any(not mask.all() for mask in covered.values()):
        raise ValueError('Some supplied sentences received no emissions.')
    return records, time.monotonic() - started


def metrics_for(gold, pred, labels):
    if len(gold) != len(pred) or not gold:
        raise ValueError('Metrics need aligned nonempty lists.')
    precision, recall, f1, support = precision_recall_fscore_support(gold, pred, labels=labels, zero_division=0)
    present = [label for label in labels if label in set(gold)]
    return {
        'macro_f1': float(f1_score(gold, pred, labels=labels, average='macro', zero_division=0)),
        'macro_precision': float(precision_score(gold, pred, labels=labels, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(gold, pred, labels=labels, average='macro', zero_division=0)),
        'present_label_macro_f1': float(f1_score(gold, pred, labels=present, average='macro', zero_division=0)),
        'accuracy': float(np.mean(np.asarray(gold) == np.asarray(pred))),
        'fixed_label_f1_ceiling': len(present) / len(labels),
        'sentences': len(gold),
        'per_label': {name: {'precision': float(precision[i]), 'recall': float(recall[i]),
                              'f1': float(f1[i]), 'support': int(support[i]), 'predicted': pred.count(name)}
                      for i, name in enumerate(labels)},
        'confusion_matrix': confusion_matrix(gold, pred, labels=labels).tolist(),
    }


@torch.no_grad()
def evaluate(model, documents, windows, labels, split, settings, output_dir=None, prefix='evaluation'):
    records, seconds = predict_emissions(model, documents, windows, labels, split, settings)
    crf = cpu_crf(model)
    label_to_id = {name: i for i, name in enumerate(labels)}
    predictions, independent = {}, {}
    golds, preds, by_domain = [], [], {'CL': ([], []), 'IT': ([], [])}
    nll, ce, weighted_ce, n, weight_sum = 0.0, 0.0, 0.0, 0, 0.0
    weights = bare(model).class_weights.detach().float().cpu()
    errors = []
    for di, emissions in records.items():
        doc = documents[di]
        tensor = torch.from_numpy(emissions)
        path = crf.decode(tensor)
        predicted = [labels[k] for k in path]
        predictions[doc['raw_id']] = predicted
        independent[doc['raw_id']] = [labels[k] for k in emissions.argmax(1)] if len(emissions) else []
        if len(predicted) != len(doc['text']):
            raise ValueError('Sentence-count mismatch after full-document decoding.')
        if doc['labels'] is None:
            continue
        gold = doc['labels']
        golds.extend(gold)
        preds.extend(predicted)
        by_domain[doc['domain']][0].extend(gold)
        by_domain[doc['domain']][1].extend(predicted)
        if len(gold):
            targets = torch.tensor([label_to_id[k] for k in gold])
            nll += float(crf.nll(tensor, targets))
            ce += float(F.cross_entropy(tensor, targets, reduction='sum'))
            weighted_ce += float(F.cross_entropy(tensor, targets, weight=weights, reduction='sum'))
            weight_sum += float(weights[targets].sum())
            n += len(gold)
        for i, (actual, guess) in enumerate(zip(gold, predicted)):
            if actual != guess and len(errors) < 100:
                errors.append({'domain': doc['domain'], 'raw_id': doc['raw_id'], 'sentence_index': i,
                    'gold': actual, 'prediction': guess, 'text': doc['text'][i][:600],
                    'previous_sentence': doc['text'][i - 1][:300] if i else None,
                    'next_sentence': doc['text'][i + 1][:300] if i + 1 < len(gold) else None})
    report = {'split': split, 'prediction_seconds': seconds, 'documents': len(records)}
    if golds:
        report['combined'] = metrics_for(golds, preds, labels)
        report['domains'] = {domain: metrics_for(g, p, labels) for domain, (g, p) in by_domain.items() if g}
        independent_flat = [guess for di in records for guess in independent[documents[di]['raw_id']]]
        report['independent_emission_macro_f1'] = metrics_for(golds, independent_flat, labels)['macro_f1']
        report['losses'] = {'document_crf_nll_per_sentence': nll / max(n, 1),
                            'unweighted_sentence_ce': ce / max(n, 1),
                            'weighted_sentence_ce': weighted_ce / max(weight_sum, 1e-12)}
    if output_dir is not None:
        output_dir = Path(output_dir)
        write_json(output_dir / (prefix + '_metrics.json'), report)
        write_json(output_dir / (prefix + '_predictions.json'), predictions)
        write_json(output_dir / (prefix + '_error_samples.json'), errors)
    return predictions, report, records


def official_check(predictions, documents, labels, evaluator_path, output_dir, prefix):
    output_dir = Path(output_dir)
    for domain in ('CL', 'IT'):
        write_json(output_dir / (prefix + '_' + domain + '_truth.json'),
                   {d['raw_id']: d['labels'] for d in documents if d['split'] == 'dev' and d['domain'] == domain})
    pred_path = output_dir / (prefix + '_prediction.json')
    write_json(pred_path, predictions)
    write_json(output_dir / 'labels.json', labels)
    result = subprocess.run([sys.executable, str(evaluator_path),
        '--truth-cl', str(output_dir / (prefix + '_CL_truth.json')),
        '--truth-it', str(output_dir / (prefix + '_IT_truth.json')),
        '--prediction', str(pred_path), '--labels', str(output_dir / 'labels.json')], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('Official evaluator failed: ' + result.stdout + result.stderr)
    (output_dir / (prefix + '_official_evaluator.txt')).write_text(result.stdout, encoding='utf-8')
    return result.stdout


def atomic_checkpoint(path, payload):
    path = Path(path)
    def tensor_bytes(value):
        if isinstance(value, torch.Tensor):
            return value.numel() * value.element_size()
        if isinstance(value, dict):
            return sum(tensor_bytes(v) for v in value.values())
        if isinstance(value, (list, tuple)):
            return sum(tensor_bytes(v) for v in value)
        return 0
    if shutil.disk_usage(path.parent).free < tensor_bytes(payload) * 1.05 + 256 * 2**20:
        raise RuntimeError('Not enough working disk for a resumable checkpoint.')
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)


def write_backup(run_dir, destination=None):
    run_dir = Path(run_dir)
    destination = Path(destination) if destination is not None else run_dir.parent / (run_dir.name + '_backup.zip')
    temporary = destination.with_suffix('.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED) as z:
        for path in sorted(run_dir.rglob('*')):
            if path.is_file() and not path.name.endswith('.tmp'):
                z.write(path, arcname=run_dir.name + '/' + str(path.relative_to(run_dir)))
    temporary.replace(destination)
    return destination


def restore_backup(supplied, run_dir):
    if supplied is None:
        return
    supplied, run_dir = Path(supplied), Path(run_dir)
    if (run_dir / 'last.pt').exists():
        return
    if supplied.suffix.lower() == '.zip':
        with zipfile.ZipFile(supplied) as z:
            lasts = [p for p in z.namelist() if Path(p).name == 'last.pt']
            if len(lasts) != 1:
                raise ValueError('Resume ZIP must contain exactly one RR last.pt.')
            parent = Path(lasts[0]).parent
            for member in z.infolist():
                relative = Path(member.filename)
                if member.is_dir() or not relative.is_relative_to(parent):
                    continue
                target = (run_dir / relative.relative_to(parent)).resolve()
                if not target.is_relative_to(run_dir.resolve()):
                    raise ValueError('Unsafe path in resume ZIP.')
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst, length=2**20)
    else:
        source = supplied.parent if supplied.is_file() else supplied
        if not (source / 'last.pt').exists():
            matches = list(source.rglob('last.pt'))
            if len(matches) != 1:
                raise ValueError('Select the exact RR resume folder or last.pt.')
            source = matches[0].parent
        for path in source.rglob('*'):
            if path.is_file():
                target = run_dir / path.relative_to(source)
                target.parent.mkdir(parents=True, exist_ok=True)
                if path.resolve() != target.resolve():
                    shutil.copy2(path, target)
    print('Restored RR resume files:', run_dir, flush=True)


def train(model, documents, train_windows, dev_windows, labels, weights, settings,
          protocol, run_dir, evaluator_path, stop_after_updates=None, teacher_targets=None):
    run_dir = Path(run_dir)
    label_to_id = {label: i for i, label in enumerate(labels)}
    n_batches = math.ceil(len(train_windows) / settings['global_window_batch'])
    updates_per_epoch = math.ceil(n_batches / settings['accumulation_steps'])
    optimizer = optimizer_for(model, settings['backbone_lr'], settings)
    scheduler = scheduler_for(optimizer, updates_per_epoch * settings['epochs'], settings)
    device = next(model.parameters()).device
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda', init_scale=128)
    state = {'epoch': 0, 'next_batch': 0, 'optimizer_updates': 0, 'skipped_updates': 0,
             'best_f1': -1.0, 'best_epoch': None, 'bad_epochs': 0, 'early_stop_reference': -1.0,
             'completed': False, 'history': [], 'epoch_crf_sum': 0.0, 'epoch_ce_sum': 0.0,
             'epoch_kd_sum': 0.0, 'epoch_sentences': 0, 'epoch_weight_sum': 0.0, 'epoch_train_seconds': 0.0}
    if settings['kd_weight'] > 0 and teacher_targets is None:
        raise ValueError('KD requires a validated trained-teacher cache.')
    last_path = run_dir / 'last.pt'
    if last_path.exists():
        saved = torch.load(last_path, map_location='cpu', weights_only=False)
        assert_protocol_matches(saved['protocol'], protocol)
        load_trainable_state(model, saved['trainable_state'])
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        scaler.load_state_dict(saved['scaler_state'])
        state = saved['training_state']
        restore_rng(saved['rng_state'])
        del saved
        print('Resumed epoch', state['epoch'] + 1, 'next batch', state['next_batch'], flush=True)
    if state['completed']:
        print('Run already complete; use its selected best checkpoint.', flush=True)
        return state
    started = last_saved = time.monotonic()
    invocation_updates = 0

    def save_last():
        atomic_checkpoint(last_path, {'protocol': protocol, 'trainable_state': trainable_state(model),
            'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(),
            'scaler_state': scaler.state_dict(), 'training_state': copy.deepcopy(state), 'rng_state': rng_state()})
        write_json(run_dir / 'history.json', state['history'])
        if settings.get('backup_on_save', True):
            write_backup(run_dir)

    def save_best(epoch, metrics, predictions):
        atomic_checkpoint(run_dir / 'best_rr.pt', {'protocol': protocol, 'trainable_state': trainable_state(model),
                                                  'epoch': epoch, 'metrics': metrics})
        write_json(run_dir / 'best_dev_metrics.json', metrics)
        write_json(run_dir / 'best_dev_predictions.json', predictions)

    try:
        if state['best_epoch'] is None:
            predictions, initial, _ = evaluate(model, documents, dev_windows, labels, 'dev', settings,
                                                run_dir, 'initial_dev')
            official_check(predictions, documents, labels, evaluator_path, run_dir, 'initial_dev')
            state['best_f1'] = state['early_stop_reference'] = initial['combined']['macro_f1']
            state['best_epoch'] = 0
            state['history'].append({'kind': 'initial_dev', 'epoch': 0, 'dev': initial})
            save_best(0, initial, predictions)
            save_last()
        while state['epoch'] < settings['epochs']:
            epoch = state['epoch']
            order = np.random.default_rng(settings['seed'] + epoch).permutation(len(train_windows))
            ordered = [train_windows[int(i)] for i in order]
            batches = [ordered[i:i + settings['global_window_batch']]
                       for i in range(0, len(ordered), settings['global_window_batch'])]
            model.train()
            for group_start in range(state['next_batch'], len(batches), settings['accumulation_steps']):
                group_started = time.monotonic()
                group = batches[group_start:group_start + settings['accumulation_steps']]
                prepared = [rr_batch(documents, part, labels, settings['chunk_tokens']) for part in group]
                if teacher_targets is not None:
                    for batch, part in zip(prepared, group):
                        batch['teacher_probs'] = batch_teacher_targets(documents, part, batch, labels, teacher_targets)
                denominators = [supervision_denominators(batch, weights) for batch in prepared]
                n_sentences = sum(n for n, _ in denominators)
                weight_sum = sum(w for _, w in denominators)
                optimizer.zero_grad(set_to_none=True)
                crf_sum, ce_sum, kd_sum = 0.0, 0.0, 0.0
                for batch in prepared:
                    stats = feed(model, batch)
                    loss = combined_loss(stats, n_sentences, weight_sum, settings['aux_ce_coefficient'], settings['kd_weight'])
                    if not torch.isfinite(loss):
                        raise RuntimeError('Nonfinite RR loss. Last complete update remains resumable.')
                    scaler.scale(loss).backward()
                    crf_sum += float(stats[:, 0].detach().sum())
                    ce_sum += float(stats[:, 1].detach().sum())
                    kd_sum += float(stats[:, 2].detach().sum())
                scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_([p for p in bare(model).parameters() if p.requires_grad], settings['gradient_clip'])
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped = scaler.get_scale() < previous_scale
                if not skipped:
                    scheduler.step()
                    state['optimizer_updates'] += 1
                    invocation_updates += 1
                state['skipped_updates'] += int(skipped)
                state['next_batch'] = group_start + len(group)
                state['epoch_crf_sum'] += crf_sum
                state['epoch_ce_sum'] += ce_sum
                state['epoch_kd_sum'] += kd_sum
                state['epoch_sentences'] += n_sentences
                state['epoch_weight_sum'] += weight_sum
                state['epoch_train_seconds'] += time.monotonic() - group_started
                row = {'kind': 'train_update', 'epoch': epoch + 1, 'next_batch': state['next_batch'],
                    'optimizer_updates': state['optimizer_updates'], 'amp_skipped': bool(skipped),
                    'window_crf_nll_per_sentence': crf_sum / n_sentences,
                    'weighted_sentence_ce': ce_sum / weight_sum,
                    'temperature_scaled_kd_per_sentence': kd_sum / n_sentences,
                    'gradient_norm': float(norm) if torch.isfinite(norm) else None,
                    'learning_rates': {g['kind']: float(g['lr']) for g in optimizer.param_groups}}
                state['history'].append(row)
                if group_start == 0 or state['optimizer_updates'] % settings['log_every_updates'] == 0:
                    print(f"Epoch {epoch+1}: batches {state['next_batch']}/{len(batches)} | "
                          f"CRF/sentence {row['window_crf_nll_per_sentence']:.4f} | "
                          f"weighted CE {row['weighted_sentence_ce']:.4f} | "
                          f"updates {state['optimizer_updates']} | skipped {state['skipped_updates']}", flush=True)
                paused = (time.monotonic() - started >= settings['max_run_hours'] * 3600 or
                          (stop_after_updates is not None and invocation_updates >= stop_after_updates))
                if paused or time.monotonic() - last_saved >= settings['save_every_minutes'] * 60:
                    save_last()
                    last_saved = time.monotonic()
                if paused:
                    print('Paused after a complete update; resume with the same RUN_ID.', flush=True)
                    return state
            predictions, dev, _ = evaluate(model, documents, dev_windows, labels, 'dev', settings,
                                            run_dir, f'epoch_{epoch+1:02d}_dev')
            training = None
            if (epoch + 1) % settings['train_metrics_every_epochs'] == 0:
                _, training, _ = evaluate(model, documents, train_windows, labels, 'train', settings,
                                          run_dir, f'epoch_{epoch+1:02d}_train')
            score = dev['combined']['macro_f1']
            if score > state['best_f1']:
                state['best_f1'], state['best_epoch'] = score, epoch + 1
                save_best(epoch + 1, dev, predictions)
            if score > state['early_stop_reference'] + settings['early_stop_min_delta']:
                state['early_stop_reference'], state['bad_epochs'] = score, 0
            else:
                state['bad_epochs'] += 1
            state['history'].append({'kind': 'epoch', 'epoch': epoch + 1, 'train': training, 'dev': dev,
                'train_update_seconds': state['epoch_train_seconds'],
                'window_crf_nll_per_sentence': state['epoch_crf_sum'] / max(state['epoch_sentences'], 1),
                'weighted_sentence_ce': state['epoch_ce_sum'] / max(state['epoch_weight_sum'], 1e-12)})
            state['history'][-1]['temperature_scaled_kd_per_sentence'] = state['epoch_kd_sum'] / max(state['epoch_sentences'], 1)
            training_text = f"{training['combined']['macro_f1']:.4f}" if training is not None else 'not evaluated this epoch'
            print(f"Epoch {epoch+1}: train F1 {training_text} | "
                  f"dev F1 {score:.4f} / ceiling {dev['combined']['fixed_label_f1_ceiling']:.4f} | "
                  f"CL {dev['domains']['CL']['macro_f1']:.4f} | IT {dev['domains']['IT']['macro_f1']:.4f} | "
                  f"best epoch {state['best_epoch']}", flush=True)
            state['epoch'] = epoch + 1
            state['next_batch'] = 0
            for key in ('epoch_crf_sum', 'epoch_ce_sum', 'epoch_kd_sum', 'epoch_weight_sum', 'epoch_train_seconds'):
                state[key] = 0.0
            state['epoch_sentences'] = 0
            state['completed'] = (state['epoch'] >= settings['epochs'] or
                (state['epoch'] >= settings['minimum_epochs'] and state['bad_epochs'] >= settings['patience']))
            save_last()
            last_saved = time.monotonic()
            if state['completed']:
                break
    except KeyboardInterrupt:
        optimizer.zero_grad(set_to_none=True)
        if not last_path.exists():
            raise
        # An interrupt could occur part-way through AdamW. Restore an atomic
        # completed checkpoint rather than serializing possibly partial updates.
        saved = torch.load(last_path, map_location='cpu', weights_only=False)
        load_trainable_state(model, saved['trainable_state'])
        state = saved['training_state']
        restore_rng(saved['rng_state'])
        del saved
        print('Interrupted; restored the latest complete saved update. Unsaved progress was discarded.', flush=True)
    finally:
        optimizer.zero_grad(set_to_none=True)
        del optimizer, scheduler, scaler
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    return state


def export_selected(model, documents, dev_windows, test_windows, labels, settings, protocol, run_dir, evaluator_path):
    run_dir = Path(run_dir)
    saved = torch.load(run_dir / 'best_rr.pt', map_location='cpu', weights_only=False)
    assert_protocol_matches(saved['protocol'], protocol)
    load_trainable_state(model, saved['trainable_state'])
    epoch = saved['epoch']
    del saved
    dev_predictions, dev, dev_emissions = evaluate(model, documents, dev_windows, labels, 'dev', settings,
                                                   run_dir, 'selected_dev')
    output = official_check(dev_predictions, documents, labels, evaluator_path, run_dir, 'selected_dev')
    match = re.search(r'Macro-F1:\s*([0-9.]+)', output)
    if match is None or abs(float(match.group(1)) - dev['combined']['macro_f1']) > 1e-6:
        raise RuntimeError('Internal and official dev metrics differ.')
    predictions, test, _ = evaluate(model, documents, test_windows, labels, 'test', settings)
    expected = {d['raw_id']: len(d['text']) for d in documents if d['split'] == 'test'}
    if set(predictions) != set(expected):
        raise ValueError('Test output document IDs differ.')
    for did, count in expected.items():
        if len(predictions[did]) != count or any(label not in labels for label in predictions[did]):
            raise ValueError('Invalid test output for ' + did)
    write_json(run_dir / 'prediction.json', predictions)
    submission = run_dir.parent / (run_dir.name + '_submission.zip')
    with zipfile.ZipFile(submission, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(run_dir / 'prediction.json', arcname='prediction.json')
    np.savez(run_dir / 'selected_dev_emissions.npz',
             **{'doc_' + str(di): array for di, array in dev_emissions.items()})
    write_json(run_dir / 'selected_dev_emission_index.json',
               {str(di): documents[di]['raw_id'] for di in dev_emissions})
    summary = {'best_epoch': epoch, 'selected_dev': dev, 'test_prediction_seconds': test['prediction_seconds'],
               'test_documents': len(expected), 'test_sentences': sum(expected.values()),
               'submission_zip': str(submission), 'checkpoint_format': 'trainable efficient RR weights; partial teacher needs original Phase 1; compact students store all trainable weights'}
    _, selected_train, _ = evaluate(model, documents,
        make_windows(documents, settings['window_sentences'], settings['overlap_sentences'], 'train'),
        labels, 'train', settings, run_dir, 'selected_train')
    summary['selected_train'] = selected_train
    write_json(run_dir / 'run_summary.json', summary)
    print(output)
    print('Selected epoch:', epoch, '| submission:', submission, flush=True)
    return summary


def sentence_hash(doc):
    return hashlib.sha256(json.dumps(doc['text'], ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()


def batch_teacher_targets(documents, windows, batch, labels, teacher_targets):
    # Only owned rows are supervised. NaN marks forbidden padding/overlap targets.
    targets = torch.full((*batch['sentence_mask'].shape, len(labels)), float('nan'))
    for b, w in enumerate(windows):
        doc = documents[w['doc_index']]
        if doc['split'] != 'train':
            raise ValueError('Teacher targets can only train on the training split.')
        q = teacher_targets[doc['raw_id']]
        if q.shape != (len(doc['text']), len(labels)):
            raise ValueError('Teacher target sentence count differs.')
        lo, hi = w['owned_start'], w['owned_end']
        targets[b, lo - w['start']:hi - w['start']] = torch.from_numpy(q[lo:hi].copy())
    return targets


def trained_teacher_cache(model, documents, labels, settings, protocol, run_dir, identity, state):
    from rr_data import file_hash
    run_dir = Path(run_dir)
    best = torch.load(run_dir / 'best_rr.pt', map_location='cpu', weights_only=False)
    assert_protocol_matches(best['protocol'], protocol)
    if best['epoch'] < 1:
        raise RuntimeError('A teacher must complete at least one training epoch before producing real KD targets.')
    load_trainable_state(model, best['trainable_state'])
    best_epoch, dev_f1 = best['epoch'], best['metrics']['combined']['macro_f1']
    del best
    records, seconds = predict_emissions(model, documents,
        make_windows(documents, settings['window_sentences'], settings['overlap_sentences'], 'train'),
        labels, 'train', settings)
    crf = cpu_crf(model)
    metadata = {'format': 'cs781_rr_teacher_targets_v1', 'purpose': 'trained_rr_teacher',
        'labels': labels, 'temperature': settings['temperature'], 'data_identity': identity,
        'teacher_run_id': run_dir.name, 'teacher_best_epoch': best_epoch,
        'teacher_dev_macro_f1': dev_f1, 'teacher_training_completed': bool(state['completed']),
        'teacher_checkpoint_sha256': file_hash(run_dir / 'best_rr.pt'),
        'teacher_protocol': protocol,
        'probability_definition': 'Full-document CRF node marginals; emissions, transitions, start and end divided by temperature.',
        'documents': []}
    arrays = {}
    for j, (di, emissions) in enumerate(records.items()):
        doc = documents[di]
        key = 'doc_' + str(j)
        arrays[key] = crf_marginals(crf, torch.from_numpy(emissions), settings['temperature']).detach().numpy().astype(np.float32)
        arrays[key + '_indices'] = np.arange(len(emissions), dtype=np.int64)
        metadata['documents'].append({'raw_id': doc['raw_id'], 'id': doc['id'], 'domain': doc['domain'],
            'split': 'train', 'sentences': len(emissions), 'text_sha256': sentence_hash(doc), 'key': key})
    path = run_dir / 'teacher_targets_train.npz'
    temporary = run_dir / 'teacher_targets_train.tmp.npz'
    np.savez_compressed(temporary, metadata=np.array(json.dumps(metadata, ensure_ascii=False)), **arrays)
    temporary.replace(path)
    write_json(run_dir / 'teacher_selection.json', {'teacher_best_epoch': best_epoch, 'teacher_dev_macro_f1': dev_f1,
        'teacher_training_completed': bool(state['completed']), 'cache_prediction_seconds': seconds,
        'cache_sha256': file_hash(path), 'documents': len(records),
        'sentences': sum(len(a) for a in records.values()), 'temperature': settings['temperature']})
    bundle = run_dir.parent / (run_dir.name + '_teacher_cache.zip')
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(path, 'teacher_targets_train.npz')
        z.write(run_dir / 'teacher_selection.json', 'teacher_selection.json')
        z.write(run_dir / 'protocol.json', 'teacher_protocol.json')
    print('TEACHER CACHE FOR THE STUDENT SESSION:', bundle, flush=True)
    return bundle


def load_teacher_cache(path, documents, labels, identity, temperature):
    from rr_data import file_hash
    with np.load(path, allow_pickle=False) as saved:
        if 'metadata' not in saved:
            raise ValueError('This is not a trained-teacher cache; preflight wiring caches are forbidden.')
        metadata = json.loads(str(saved['metadata'].item()))
        if metadata.get('format') != 'cs781_rr_teacher_targets_v1' or metadata.get('purpose') != 'trained_rr_teacher':
            raise ValueError('A trained RR teacher cache is required.')
        if metadata['labels'] != labels or metadata['temperature'] != temperature or metadata['data_identity'] != identity:
            raise ValueError('Teacher label order, temperature or official data identity differs.')
        if metadata['teacher_best_epoch'] < 1 or not metadata.get('teacher_checkpoint_sha256'):
            raise ValueError('Teacher provenance is missing.')
        expected = {d['raw_id']: d for d in documents if d['split'] == 'train'}
        if len(metadata['documents']) != len(expected) or {r['raw_id'] for r in metadata['documents']} != set(expected):
            raise ValueError('Teacher cache must cover each training document exactly once.')
        targets = {}
        for r in metadata['documents']:
            doc = expected[r['raw_id']]
            if r['split'] != 'train' or r['id'] != doc['id'] or r['text_sha256'] != sentence_hash(doc):
                raise ValueError('Teacher document/sentence alignment differs.')
            q = saved[r['key']].copy()
            index = saved[r['key'] + '_indices']
            if q.shape != (len(doc['text']), len(labels)) or not np.array_equal(index, np.arange(len(doc['text']))):
                raise ValueError('Teacher probability dimensions or sentence indices differ.')
            if not np.isfinite(q).all() or (q < 0).any() or (q > 1 + 1e-6).any() or not np.allclose(q.sum(-1), 1, atol=2e-5):
                raise ValueError('Invalid teacher probabilities.')
            targets[r['raw_id']] = q.astype(np.float32)
    return targets, metadata, file_hash(path)


def save_diagnostics(run_dir):
    run_dir = Path(run_dir)
    path = run_dir.parent / (run_dir.name + '_diagnostics.zip')
    with zipfile.ZipFile(path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for f in sorted(run_dir.rglob('*')):
            if f.is_file() and f.suffix in ('.json', '.csv', '.png', '.py', '.txt'):
                z.write(f, arcname=run_dir.name + '/' + str(f.relative_to(run_dir)))
    try:
        backup = write_backup(run_dir)
    except (OSError, RuntimeError) as error:
        backup = None
        write_json(run_dir / 'backup_status.json', {'status': 'failed', 'failure': str(error),
            'note': 'Checkpoint files remain in the run directory.'})
        print('Backup ZIP could not be written; checkpoint files remain on disk:', error, flush=True)
    return path, backup


def summarize_run(run_dir, state):
    import csv
    run_dir = Path(run_dir)
    rows = []
    for row in state['history']:
        if row['kind'] != 'epoch':
            continue
        train_metric = row.get('train')
        rows.append({'epoch': row['epoch'], 'dev_macro_f1': row['dev']['combined']['macro_f1'],
            'train_macro_f1': train_metric['combined']['macro_f1'] if train_metric else None,
            'dev_ce': row['dev']['losses']['unweighted_sentence_ce'],
            'window_crf_nll_per_sentence': row['window_crf_nll_per_sentence'],
            'weighted_sentence_ce': row['weighted_sentence_ce'],
            'temperature_scaled_kd_per_sentence': row.get('temperature_scaled_kd_per_sentence', 0),
            'train_update_minutes': row['train_update_seconds'] / 60})
    if rows:
        with open(run_dir / 'epoch_history.csv', 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader(); writer.writerows(rows)
        try:
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, 2, figsize=(11, 4))
            axes[0].plot([r['epoch'] for r in rows], [r['dev_macro_f1'] for r in rows], label='Dev')
            known = [r for r in rows if r['train_macro_f1'] is not None]
            axes[0].plot([r['epoch'] for r in known], [r['train_macro_f1'] for r in known], label='Train')
            axes[0].set(xlabel='Epoch', ylabel='Official 13-label macro-F1'); axes[0].legend()
            axes[1].plot([r['epoch'] for r in rows], [r['dev_ce'] for r in rows], label='Dev CE')
            axes[1].plot([r['epoch'] for r in rows], [r['weighted_sentence_ce'] for r in rows], label='Train weighted CE')
            axes[1].set(xlabel='Epoch', ylabel='Loss'); axes[1].legend()
            fig.tight_layout(); fig.savefig(run_dir / 'training_curves.png', dpi=140); plt.close(fig)
        except ImportError:
            pass
    write_json(run_dir / 'training_state_summary.json', {k: v for k, v in state.items() if k != 'history'})


def model_fingerprint(model):
    digest = hashlib.sha256()
    for name, parameter in bare(model).named_parameters():
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_efficient_engine.py


In [12]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_efficient_workflow.py
"""Teacher and compact gold/KD stages, verification and file handoff."""
import copy
import gc
import hashlib
import json
from pathlib import Path
import shutil
import time
import traceback
import zipfile
import numpy as np
import torch
from torch import nn
import rr_data
from rr_data import (write_json, read_json, file_hash, find_input, load_own_checkpoint,
    prepare_rr_data, load_rr_documents, load_tokenizer, tokenize_documents, class_weights, make_windows)
from rr_shared import bare, load_trainable_state
from architecture_models import ModelConfig, EfficientLM
from rr_efficient_model import compact_lm, EfficientRR, dropout_audit
from rr_efficient_probe import run_preflight, parallel_check, seed_all
from rr_efficient_engine import (train, export_selected, trained_teacher_cache, load_teacher_cache,
    restore_backup, assert_protocol_matches, save_diagnostics, summarize_run, model_fingerprint,
    rng_state, restore_rng)

PREFLIGHT_MODULES = ['phase1_utils.py', 'a1_tokenizer.py', 'architecture_models.py', 'rr_data.py',
    'rr_shared.py', 'rr_check_utils.py', 'rr_efficient_model.py', 'rr_efficient_probe.py']


def load_context(root, src, prep_input, efficient_input, data_input, reference):
    root, src = Path(root), Path(src)
    rr_data.ROOT = root
    rr_data.INPUT_ROOTS = [Path('/kaggle/input')]
    data = prepare_rr_data(root / 'official_data/Phase_3_RR', reference['rr_data_hashes'], reference['rr_data_revision'], data_input)
    labels, documents = load_rr_documents(data)
    tokenizer_path = find_input('tokenizer_state.pkl', prep_input, hint='prep_002_mixed16k' if prep_input is None else None)
    tokenizer, signature, manifest = load_tokenizer(tokenizer_path)
    if file_hash(tokenizer_path) != reference['tokenizer_sha256']:
        raise ValueError('Prep tokenizer differs from the uploaded preflight identity.')
    fingerprint = hashlib.sha256(json.dumps({'data_hashes': reference['rr_data_hashes'],
        'tokenizer_sha256': reference['tokenizer_sha256']}, sort_keys=True).encode()).hexdigest()
    tokenize_documents(documents, tokenizer, fingerprint, root / 'token_cache')
    checkpoint = find_input('best_model.pt', efficient_input, hint='efficient' if efficient_input is None else None)
    if file_hash(checkpoint) != reference['phase1_checkpoint_sha256']:
        raise ValueError('Select the original efficient_002 500M checkpoint used by the uploaded preflight.')
    native = checkpoint.parent / 'sources/architecture_models.py'
    if not native.exists() or file_hash(native) != reference['architecture_source_sha256']:
        raise ValueError('Original efficient architecture source differs from the reviewed 500M source.')
    if file_hash(src / 'architecture_models.py') != reference['architecture_source_sha256']:
        raise ValueError('Embedded original efficient architecture source was changed.')
    saved = load_own_checkpoint(checkpoint)
    if saved['tokenizer_signature'] != signature or saved['model_config'] != reference['model_config']:
        raise ValueError('Checkpoint configuration or tokenizer signature differs.')
    cfg = ModelConfig(**saved['model_config'])
    if cfg.architecture != 'efficient' or cfg.n_layers != 16:
        raise ValueError('Use the original efficient Phase 1 LM, not a task-trained student.')
    if not torch.equal(saved['model_state']['head.weight'], saved['model_state']['tok_emb.weight']):
        raise ValueError('The original tied LM weights disagree.')
    original = EfficientLM(cfg)
    original.load_state_dict(saved['model_state'], strict=True)
    del saved
    weights, counts = class_weights(documents, labels, 5.0)
    expected = {'train': 16845, 'dev': 2142, 'test': 2197}
    if {s: sum(len(d['text']) for d in documents if d['split'] == s) for s in expected} != expected:
        raise ValueError('Official RR sentence counts differ.')
    identity = {'rr_data_revision': reference['rr_data_revision'], 'rr_data_hashes': reference['rr_data_hashes'],
                'tokenizer_sha256': reference['tokenizer_sha256']}
    return {'root': root, 'src': src, 'documents': documents, 'labels': labels, 'weights': weights,
        'counts': counts, 'original': original, 'tokenizer_path': tokenizer_path,
        'phase1_checkpoint_path': checkpoint, 'reference': reference, 'data_identity': identity,
        'source_sha256': {p.name: file_hash(p) for p in src.glob('*.py')}}


def usable_preflight(folder, context, settings):
    folder = Path(folder)
    try:
        report, inputs = read_json(folder / 'preflight_report.json'), read_json(folder / 'input_record.json')
        if report['status'] != 'passed':
            return None
        if inputs['phase1_checkpoint_sha256'] != context['reference']['phase1_checkpoint_sha256']:
            return None
        if inputs['tokenizer_sha256'] != context['reference']['tokenizer_sha256']:
            return None
        if any(inputs['source_sha256'].get(n) != context['source_sha256'][n] for n in PREFLIGHT_MODULES):
            return None
        keys = ['window_sentences', 'overlap_sentences', 'chunk_tokens', 'chunk_microbatch', 'global_window_batch',
            'accumulation_steps', 'gradient_checkpointing', 'projection_dim', 'context_hidden', 'head_dropout', 'backbone_dropout',
            'pooling', 'student_layers', 'temperature', 'aux_ce_coefficient', 'teacher_lr', 'student_lr', 'head_lr', 'crf_lr']
        if any(report['settings'].get(k) != settings[k] for k in keys):
            return None
        environment = read_json(folder / 'environment.json')
        if environment['torch'] != torch.__version__:
            return None
        names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        if [d['name'] for d in environment['gpus']] != names:
            return None
        return report
    except (KeyError, FileNotFoundError, ValueError, TypeError):
        return None


def ensure_preflight(context, settings, supplied=None):
    root = context['root'] / 'preflights' / 'efficient_rr_heavy_preflight_001'
    if supplied is not None:
        p = find_input('preflight_report.json', supplied, required=False)
        if p is not None:
            supplied_path = Path(supplied)
            # find_input extracts a requested member only. Bring its proof companions too.
            if supplied_path.is_file() and supplied_path.suffix.lower() == '.zip':
                with zipfile.ZipFile(supplied_path) as z:
                    reports = [n for n in z.namelist() if Path(n).name == 'preflight_report.json']
                    if len(reports) == 1:
                        parent = str(Path(reports[0]).parent)
                        for name in ('input_record.json', 'environment.json'):
                            member = parent + '/' + name
                            if member in z.namelist():
                                (p.parent / name).write_bytes(z.read(member))
            found = usable_preflight(p.parent, context, settings)
            if found is not None:
                print('Reused passed source/data/GPU-matched preflight:', p.parent, flush=True)
                return found, p.parent
    if root.exists():
        for path in sorted(root.glob('*/preflight_report.json'), reverse=True):
            found = usable_preflight(path.parent, context, settings)
            if found is not None:
                print('Reused passed corrected preflight:', path.parent, flush=True)
                return found, path.parent
    attempt = root / time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
    attempt.mkdir(parents=True, exist_ok=True)
    (attempt / 'sources').mkdir(exist_ok=True)
    for name in PREFLIGHT_MODULES:
        shutil.copy2(context['src'] / name, attempt / 'sources' / name)
    record = dict(context['reference'], source_sha256=context['source_sha256'], settings=settings)
    write_json(attempt / 'input_record.json', record)
    write_json(attempt / 'environment.json', {'torch': torch.__version__,
        'gpus': [{'name': torch.cuda.get_device_name(i),
                  'total_gib': torch.cuda.get_device_properties(i).total_memory / 2**30} for i in range(2)]})
    print('Running fresh preflight for the heavier effective dropout settings.', flush=True)
    report, diagnostics = run_preflight(context['original'], context['documents'], context['labels'],
        context['weights'], settings, attempt)
    destination = Path('/kaggle/working') / 'efficient_rr_heavy_preflight_001_diagnostics.zip'
    shutil.copy2(diagnostics, destination)
    print('CORRECTED PREFLIGHT:', report['status'], '| diagnostics:', destination, flush=True)
    if report['status'] != 'passed':
        raise RuntimeError('Corrected preflight did not pass; full training was not started. Send ' + str(destination))
    return report, attempt


def choose_settings(stage, context, common, training, operations, preflight, teacher_adaptation, saved=None):
    if saved is not None:
        result = dict(saved['settings'])
        if saved['stage'] != stage:
            raise ValueError('Resume stage differs.')
        if stage == 'teacher' and teacher_adaptation != 'auto' and saved['adaptation'] != teacher_adaptation:
            raise ValueError('Requested teacher adaptation differs from the saved run.')
        # The configuration remains fixed on resume; operational pause/log/save intervals can change.
        requested = dict(common)
        for key in ('seed','window_sentences','overlap_sentences','chunk_tokens','global_window_batch','accumulation_steps',
                    'gradient_checkpointing','projection_dim','context_hidden','head_dropout','backbone_dropout','pooling','student_layers',
                    'head_lr','crf_lr','weight_decay','gradient_clip','class_weight_cap','aux_ce_coefficient','temperature'):
            prior = result.get(key, context['reference']['model_config']['dropout']) if key == 'backbone_dropout' else result[key]
            if prior != requested[key]:
                raise ValueError('Saved model/training setting differs: ' + key)
        return dict(result, **operations), saved['adaptation']
    passed = [c for c in preflight['cases'] if c['status'] == 'passed']
    if stage == 'teacher':
        candidates = [c for c in passed if c['name'].startswith('teacher')]
        if teacher_adaptation != 'auto':
            candidates = [c for c in candidates if c['name'] == 'teacher_' + teacher_adaptation]
        if not candidates:
            raise RuntimeError('The requested teacher configuration did not pass preflight. Use auto or a passing adaptation.')
        fitting = [c for c in candidates if c['timing']['estimated_25_epoch_hours'] <= operations['max_run_hours']]
        selected = (fitting or sorted(candidates, key=lambda c: c['timing']['estimated_25_epoch_hours']))[0]
        adaptation = 'upper4' if selected['name'] == 'teacher_upper4' else 'full'
    else:
        selected = next(c for c in passed if c['name'] == 'student')
        adaptation = 'full'
    result = dict(common, **training, **operations)
    result.update(chunk_microbatch=selected['chunk_microbatch'],
        backbone_lr=common['teacher_lr'] if stage == 'teacher' else common['student_lr'],
        kd_weight=common['kd_weight'] if stage == 'student_kd' else 0.0)
    return result, adaptation


def stage_protocol(stage, context, settings, adaptation, cache_sha):
    operational = {'max_run_hours', 'save_every_minutes', 'log_every_updates', 'backup_on_save'}
    return {'stage': stage, 'method': 'efficient_rr_gold_plus_crf_marginal_kd' if stage == 'student_kd' else
        ('efficient_rr_supervised_compact_baseline' if stage == 'student_baseline' else 'efficient_rr_' + adaptation + '_fine_tuning'),
        'phase1_checkpoint_sha256': context['reference']['phase1_checkpoint_sha256'],
        'phase1_model_config': context['reference']['model_config'], 'data_identity': context['data_identity'],
        'labels': context['labels'], 'training_counts': context['counts'], 'class_weights': context['weights'].tolist(),
        'adaptation': adaptation, 'settings': {k: v for k, v in settings.items() if k not in operational},
        'teacher_cache_sha256': cache_sha,
        'source_sha256': context['source_sha256']}


def release_model(model):
    if model is not None:
        model.zero_grad(set_to_none=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_stage(stage, run_id, context, common, training, operations, preflight, proof_dir,
              teacher_adaptation='auto', resume_input=None, teacher_cache_input=None, stop_after_updates=None):
    run_dir = context['root'] / 'runs' / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    restore_backup(resume_input, run_dir)
    has_checkpoint = (run_dir / 'last.pt').exists() or (run_dir / 'best_rr.pt').exists()
    saved_protocol = read_json(run_dir / 'protocol.json') if has_checkpoint and (run_dir / 'protocol.json').exists() else None
    if not has_checkpoint:
        (run_dir / 'initialization.json').unlink(missing_ok=True)
    settings, adaptation = choose_settings(stage, context, common, training, operations, preflight,
        teacher_adaptation, saved_protocol)
    teacher_targets = None
    teacher_metadata = None
    cache_sha = None
    if stage == 'student_kd':
        supplied = teacher_cache_input
        if supplied is None:
            local = context['root'] / 'runs' / ('efficient_rr_teacher_' + run_id.rsplit('_', 1)[-1]) / 'teacher_targets_train.npz'
            path = local if local.exists() else find_input('teacher_targets_train.npz', required=True)
        else:
            path = find_input('teacher_targets_train.npz', supplied)
        teacher_targets, teacher_metadata, cache_sha = load_teacher_cache(path, context['documents'], context['labels'],
            context['data_identity'], settings['temperature'])
        write_json(run_dir / 'teacher_identity.json', {'cache_sha256': cache_sha,
            'teacher_run_id': teacher_metadata['teacher_run_id'], 'teacher_best_epoch': teacher_metadata['teacher_best_epoch'],
            'teacher_dev_macro_f1': teacher_metadata['teacher_dev_macro_f1'],
            'teacher_checkpoint_sha256': teacher_metadata['teacher_checkpoint_sha256']})
    protocol = stage_protocol(stage, context, settings, adaptation, cache_sha)
    if saved_protocol is not None:
        assert_protocol_matches(saved_protocol, protocol)
    write_json(run_dir / 'protocol.json', protocol)
    write_json(run_dir / 'operational_settings.json', operations)
    (run_dir / 'sources').mkdir(exist_ok=True)
    for path in context['src'].glob('*.py'):
        shutil.copy2(path, run_dir / 'sources' / path.name)
    (run_dir / 'preflight_evidence').mkdir(exist_ok=True)
    for name in ('preflight_report.json', 'input_record.json', 'environment.json'):
        shutil.copy2(Path(proof_dir) / name, run_dir / 'preflight_evidence' / name)
    seed_all(settings['seed'])
    lm = copy.deepcopy(context['original']) if stage == 'teacher' else compact_lm(context['original'], tuple(settings['student_layers']))
    base = EfficientRR(lm, len(context['labels']), context['weights'], settings, adaptation)
    del lm
    effective_dropout = dropout_audit(base)
    write_json(run_dir / 'effective_dropout.json', effective_dropout)
    print('Effective training dropout:', effective_dropout, flush=True)
    initial_hash = model_fingerprint(base)
    initial_path = run_dir / 'initialization.json'
    initial_record = {'phase1_checkpoint_sha256': context['reference']['phase1_checkpoint_sha256'],
        'initial_all_parameters_sha256': initial_hash, 'student_blocks': settings['student_layers'] if stage != 'teacher' else None,
        'total_parameters': sum(p.numel() for p in base.parameters()),
        'trainable_parameters': sum(p.numel() for p in base.parameters() if p.requires_grad)}
    if initial_path.exists() and read_json(initial_path) != initial_record:
        raise ValueError('Fresh Phase 1/head initialization differs from the saved run.')
    write_json(initial_path, initial_record)
    if stage != 'teacher':
        partner = run_id.replace('student_kd_', 'student_gold_') if stage == 'student_kd' else run_id.replace('student_gold_', 'student_kd_')
        partner_path = run_dir.parent / partner / 'initialization.json'
        if partner_path.exists() and read_json(partner_path)['initial_all_parameters_sha256'] != initial_hash:
            raise ValueError('Gold-only and KD student initializations differ.')
    base = base.to('cuda:0')
    model = nn.DataParallel(base, device_ids=[0, 1])
    state = None
    summary = None
    try:
        startup_rng = rng_state()
        try:
            startup = parallel_check(model, context['documents'], context['labels'], context['weights'], settings)
        finally:
            model.zero_grad(set_to_none=True)
            restore_rng(startup_rng)
        write_json(run_dir / 'startup_check.json', startup)
        print('\nRUN:', run_id, '| adaptation:', adaptation, '| epoch cap:', settings['epochs'],
            '| patience:', settings['patience'], '| wall-time hours:', settings['max_run_hours'], flush=True)
        train_windows = make_windows(context['documents'], settings['window_sentences'], settings['overlap_sentences'], 'train')
        dev_windows = make_windows(context['documents'], settings['window_sentences'], settings['overlap_sentences'], 'dev')
        test_windows = make_windows(context['documents'], settings['window_sentences'], settings['overlap_sentences'], 'test')
        state = train(model, context['documents'], train_windows, dev_windows, context['labels'], context['weights'],
            settings, protocol, run_dir, context['src'] / 'evaluate_phase3.py', stop_after_updates=stop_after_updates,
            teacher_targets=teacher_targets)
        summarize_run(run_dir, state)
        summary = export_selected(model, context['documents'], dev_windows, test_windows, context['labels'],
            settings, protocol, run_dir, context['src'] / 'evaluate_phase3.py')
        summary['training_completed'] = bool(state['completed'])
        summary['selected_after_partial_training'] = not bool(state['completed'])
        summary['total_parameters'] = initial_record['total_parameters']
        summary['trainable_parameters'] = initial_record['trainable_parameters']
        write_json(run_dir / 'run_summary.json', summary)
        if stage == 'teacher' and summary['best_epoch'] >= 1:
            bundle = trained_teacher_cache(model, context['documents'], context['labels'], settings, protocol,
                run_dir, context['data_identity'], state)
            with zipfile.ZipFile(bundle, 'a', zipfile.ZIP_DEFLATED) as z:
                for name in ('preflight_report.json', 'input_record.json', 'environment.json'):
                    z.write(Path(proof_dir) / name, 'preflight_evidence/' + name)
            summary['teacher_cache_zip'] = str(bundle)
            write_json(run_dir / 'run_summary.json', summary)
        if stage != 'teacher':
            # All retained student weights are present in best_rr.pt. Include the tokenizer for inference.
            portable = run_dir.parent / (run_dir.name + '_inference.zip')
            with zipfile.ZipFile(portable, 'w', zipfile.ZIP_STORED) as z:
                z.write(run_dir / 'best_rr.pt', 'best_rr.pt')
                z.write(run_dir / 'protocol.json', 'protocol.json')
                z.write(context['tokenizer_path'], 'tokenizer_state.pkl')
                z.write(context['tokenizer_path'].parent / 'cache_manifest.json', 'cache_manifest.json')
                for path in (run_dir / 'sources').glob('*.py'):
                    z.write(path, 'sources/' + path.name)
                z.writestr('README.txt', 'Student inference weights: best_rr.pt contains all named trainable parameters.\n'
                    'Construct EfficientLM with phase1_model_config and n_layers=len(student_layers), then EfficientRR,\n'
                    'and load rr_shared.load_trainable_state(model, checkpoint["trainable_state"]).\n'
                    'Use the included tokenizer/manifest and labels in protocol.json. No Phase 1 checkpoint is needed for inference.\n')
            summary['inference_zip'] = str(portable)
        write_json(run_dir / 'run_summary.json', summary)
        return summary
    except BaseException as error:
        write_json(run_dir / 'failure.json', {'failure': str(error), 'traceback': traceback.format_exc(),
            'note': 'Only completed atomic saved updates are resumable; no failed/partial optimizer update was saved.'})
        raise
    finally:
        if state is not None:
            summarize_run(run_dir, state)
        diagnostics, backup = save_diagnostics(run_dir)
        print('DIAGNOSTICS:', diagnostics, '\nBACKUP:', backup, flush=True)
        model.zero_grad(set_to_none=True)
        del model, base
        release_model(None)


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/rr_efficient_workflow.py


In [13]:
%%writefile /kaggle/working/cs781_phase3_efficient_rr_heavy_src/evaluate_phase3.py
#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.

Usage:
  python evaluate_phase3.py \
      --truth-cl cl_test_truth.json \
      --truth-it it_test_truth.json \
      --prediction prediction.json \
      --labels labels.json

Each truth file must be a JSON object mapping document IDs to ordered lists of
sentence labels. prediction.json must have the same structure and contain
predictions for documents from both reference files.

labels.json may be either:
  ["Label1", "Label2", ...]
or a label->id / id->label mapping.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            # label -> integer ID
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            # integer/string ID -> label
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument(
        "--truth-cl",
        required=True,
        help="Local held-out reference JSON for the CL split.",
    )
    parser.add_argument(
        "--truth-it",
        required=True,
        help="Local held-out reference JSON for the IT split.",
    )
    parser.add_argument(
        "--prediction",
        required=True,
        help="Participant prediction JSON covering documents from both splits.",
    )
    parser.add_argument(
        "--labels",
        required=True,
        help="Public Phase 3 label vocabulary JSON.",
    )
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [
            f"missing={len(missing)}",
            f"extra={len(extra)}",
        ]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)


Writing /kaggle/working/cs781_phase3_efficient_rr_heavy_src/evaluate_phase3.py


## 3. Original Phase 1 inputs and official data

The notebook verifies the seven `Phase_3_RR` files at the pinned revision and reuses the 16K byte BPE tokenizer from Phase 1. I trained on the official CL and IT training sets together and kept the official document-level dev and test splits. The scorer uses a fixed vocabulary of 13 roles. Two roles are absent from combined dev, giving a perfect-prediction dev ceiling of 11/13.

The token model processes chunks of up to 256 tokens, including BOS and EOS. Longer sentences retain all their chunks, which are combined by content length. A bidirectional GRU contextualizes the sentence vectors in 128-sentence windows with 32-sentence overlap. Each sentence contributes supervised loss and inference emissions through one owned window. I reassemble those emissions in order and decode one CRF path per document.

This run verifies the original Phase 1 efficient checkpoint and its native source. The selected checkpoint came from the continuation run at 405.81M input positions.


In [14]:
REFERENCE = {'phase1_checkpoint_sha256': '7117b452f4ba76bf8590a633c6b2bfe7472a328aefed7127de171d73b68bf3ea', 'tokenizer_sha256': '080c6d2ce49a5c9ccfdd707145ec2b49c8a3f3d2e676e30d9c6e93c948900d9c', 'rr_data_revision': '89edba6a44d6bb9593fd13b6895b5cd3b06eb82e', 'rr_data_hashes': {'CL_train.json': '2fdad020443ad33734a8c3a59f4fbae72582e0535692708ad902473087333ed1', 'CL_dev.json': '95ec2dac8b4c869f9fd1ae434546103b93d9c0a5340b2e5a7e611773519dbbcc', 'IT_train.json': 'a9111869c768c1adfb492ba03df72b14d77fcce9f468a2a167ccf9d29a3b010b', 'IT_dev.json': '825e158bfcb46c1001dc0062d43bdd8f4060c5214ec465aedd2c58d28da1b20a', 'label_vocab.json': 'd4d0175f792a5349ccc028ef3a1d220374169e999550441e84378dcf39e9ae45', 'CL_test.json': '33547ab29f5ae9afda36d92d779aa955676ae1d31df0ca479f09abfeeb48f248', 'IT_test.json': '3117dbe4a909bdec4628c200ed8ed2617a651d633296b41977b00f8b556dbc81'}, 'architecture_source_sha256': '4f2db692383361b59ca1e606ef8cddebb9d1d64fa7ace806b7d5766a735c9937', 'model_config': {'architecture': 'efficient', 'vocab_size': 16000, 'd_model': 1024, 'n_heads': 16, 'n_layers': 16, 'n_kv_heads': 4, 'd_ff': 4864, 'max_seq_len': 2560, 'dropout': 0.1, 'window_size': 512, 'query_chunk_size': 128}}
sys.path.insert(0, str(SRC))
MODULES = ['phase1_utils','a1_tokenizer','architecture_models','rr_data','rr_shared','rr_check_utils',
           'rr_efficient_model','rr_efficient_probe','rr_efficient_engine','rr_efficient_workflow']
for name in MODULES:
    sys.modules.pop(name, None)
from rr_data import write_json, read_json
from rr_efficient_workflow import load_context, ensure_preflight, run_stage
from rr_efficient_probe import seed_all

seed_all(COMMON['seed'])
CONTEXT = load_context(ROOT, SRC, PREP_INPUT, EFFICIENT_INPUT, RR_DATA_INPUT, REFERENCE)
COMMON['cache_identity'] = hashlib.sha256(json.dumps(REFERENCE, sort_keys=True).encode()).hexdigest()
display(pd.DataFrame({'label':CONTEXT['labels'], 'training_sentences':CONTEXT['counts'],
                      'class_weight':CONTEXT['weights'].tolist()}))
print('Verified Phase 1 model:', CONTEXT['phase1_checkpoint_path'])
print('Backbone parameters:', sum(p.numel() for p in CONTEXT['original'].parameters()))
print('No NER model is loaded. RR heads start fresh.')


Loaded verified RR token cache.


,label,training_sentences,class_weight
0,Fact,4098,1.024469
1,Issue,107,5.000000
2,ArgumentPetitioner,1613,1.632930
3,ArgumentRespondent,1348,1.786240
4,Statute,1099,1.978271
5,Dissent,284,3.891576
6,PrecedentReliedUpon,1945,1.487048
7,PrecedentNotReliedUpon,569,2.749341
8,PrecedentOverruled,3,5.000000
9,RulingByLowerCourt,874,2.218345


Verified Phase 1 model: /kaggle/input/datasets/ash9201k/efficient-002/cs781_phase1/runs/efficient_002_mixed217m_500m/best_model.pt
Backbone parameters: 217872384
No NER model is loaded. RR heads start fresh.


## 4. Verify effective dropout and GPU resources

The model applies dropout to attention probabilities and to the residual and feed-forward paths during training. The preflight checks the effective values and tests the full teacher with this configuration before applying real updates.


In [15]:
PREFLIGHT_REPORT, PROOF_DIR = ensure_preflight(CONTEXT, COMMON, PREFLIGHT_INPUT)
display(pd.DataFrame([{
    'case':c['name'], 'status':c['status'], 'chunk_microbatch':c.get('chunk_microbatch'),
    'trainable_M':c.get('parameters',{}).get('trainable',0)/1e6,
    'backbone_dropout':c.get('effective_dropout',{}).get('backbone_dropout'),
    'head_dropout':c.get('effective_dropout',{}).get('head_dropout'),
    'peak_allocated_GiB':c.get('peak_allocated_gib'),
    'estimated_35_epochs_h':c.get('timing',{}).get('estimated_35_epoch_hours'),
} for c in PREFLIGHT_REPORT['cases']]))
if not any(c['name']=='teacher_full' and c['status']=='passed' for c in PREFLIGHT_REPORT['cases']):
    raise RuntimeError('The full teacher did not pass. Retain the new preflight diagnostics for review.')
print('Passed. Next cell starts the new full teacher; the invocation budget may pause it before 35 epochs.')


Running fresh preflight for the heavier effective dropout settings.

Starting teacher_full | chunk microbatch 8
teacher_full | passed

Starting teacher_upper4 | chunk microbatch 8
teacher_upper4 | passed

Starting student | chunk microbatch 8
Disposable student learning: 1 / 40 | loss 3.9434
Disposable student learning: 10 / 40 | loss 1.4325
Disposable student learning: 20 / 40 | loss 0.1597
Disposable student learning: 30 / 40 | loss 0.0157
Disposable student learning: 40 / 40 | loss 0.0039
student | passed
CORRECTED PREFLIGHT: passed | diagnostics: /kaggle/working/efficient_rr_heavy_preflight_001_diagnostics.zip


,case,status,chunk_microbatch,trainable_M,backbone_dropout,head_dropout,peak_allocated_GiB,estimated_35_epochs_h
0,teacher_full,passed,8,218.697936,0.2,0.4,"[6.11425256729126, 3.603987216949463]",4.361726
1,teacher_upper4,passed,8,51.199184,0.2,0.4,"[3.1751112937927246, 1.4848213195800781]",2.452796
2,student,passed,8,54.990288,0.2,0.4,"[1.6310787200927734, 0.9512476921081543]",1.259927


Passed. Next cell starts the new full teacher; the invocation budget may pause it before 35 epochs.


## 5. Train the regularized teacher

I started from the same Phase 1 weights with fresh RR components. The recorded invocation had a six-hour training cap. I stopped after 18 completed epochs and retained the best dev checkpoint from epoch 16.


In [16]:
operations = {
    'max_run_hours':RUN_BUDGET_HOURS['teacher'], 'save_every_minutes':SAVE_EVERY_MINUTES,
    'log_every_updates':LOG_EVERY_UPDATES, 'backup_on_save':False,
}
selected_training = dict(TRAINING, epochs=EPOCH_CAPS['teacher'])
RESULT = run_stage('teacher', RUN_IDS['teacher'], CONTEXT, COMMON,
    selected_training, operations, PREFLIGHT_REPORT, PROOF_DIR,
    teacher_adaptation=TEACHER_ADAPTATION, resume_input=RESUME_INPUT,
    teacher_cache_input=None, stop_after_updates=STOP_AFTER_UPDATES)
gc.collect()
torch.cuda.empty_cache()
print('Finished or paused safely. Check training_completed in the summary.')


Effective training dropout: {'status': 'passed', 'backbone_dropout': 0.2, 'head_dropout': 0.4, 'attention_probability_modules': 16, 'attention_residual_and_ffn_dropout_modules': 32, 'backbone_configuration_dropout': 0.2, 'original_phase1_parameter_values_preserved_by_dropout_change': True}

RUN: efficient_rr_teacher_heavyreg_001 | adaptation: full | epoch cap: 35 | patience: 5 | wall-time hours: 6
Epoch 1: batches 2/99 | CRF/sentence 2.4425 | weighted CE 2.4657 | updates 1 | skipped 0
Epoch 1: batches 20/99 | CRF/sentence 2.3691 | weighted CE 2.4443 | updates 10 | skipped 0
Epoch 1: batches 40/99 | CRF/sentence 2.2687 | weighted CE 2.4233 | updates 20 | skipped 0
Epoch 1: batches 60/99 | CRF/sentence 2.1001 | weighted CE 2.2442 | updates 30 | skipped 0
Epoch 1: batches 80/99 | CRF/sentence 2.0788 | weighted CE 2.3408 | updates 40 | skipped 0
Epoch 1: batches 99/99 | CRF/sentence 2.3283 | weighted CE 2.6239 | updates 50 | skipped 0
Epoch 1: train F1 not evaluated this epoch | dev F1 0.0

KeyboardInterrupt: 

## 6. Export and recorded interruption

The saved run wrote its selected test predictions and submission ZIP before the final train-set evaluation was interrupted. The comparison cell expects the final `run_summary.json`, which explains the saved missing-file traceback. The dev scores above come from the training log and the test score comes from CodaBench.


In [17]:
BASELINE_SUMMARY = {'run_id': 'efficient_rr_teacher_001', 'best_epoch': 13, 'train_f1': 0.860439432606496, 'dev_f1': 0.44357942934840733, 'CL_dev_f1': 0.35835885765223724, 'IT_dev_f1': 0.45128907409973706, 'codabench_f1': 0.4408, 'training_completed': True, 'initial_parameters_sha256': '03504553a6bc1e1e1d83e69b5a6774d4b767157591ec3856c220c50302b0999e'}
BASELINE_SETTINGS = {'seed': 42, 'window_sentences': 128, 'overlap_sentences': 32, 'chunk_tokens': 256, 'chunk_microbatch': 8, 'global_window_batch': 2, 'accumulation_steps': 2, 'gradient_checkpointing': True, 'projection_dim': 256, 'context_hidden': 128, 'head_dropout': 0.2, 'pooling': 'mean_last', 'student_layers': [0, 7, 15], 'teacher_lr': 1e-05, 'student_lr': 2e-05, 'head_lr': 0.0003, 'crf_lr': 0.001, 'weight_decay': 0.01, 'gradient_clip': 1.0, 'class_weight_cap': 5.0, 'aux_ce_coefficient': 0.5, 'temperature': 2.0, 'kd_weight': 0.0, 'learning_steps': 40, 'teacher_training_hours': 2.75, 'student_training_hours': 5.0, 'cache_identity': '5a57bc88ad732e5fa4fdcf2eef0409f4b64926cd865006418b0a4c3a706cba56', 'minimum_epochs': 5, 'patience': 5, 'early_stop_min_delta': 0.0001, 'warmup_fraction': 0.05, 'minimum_lr_fraction': 0.01, 'train_metrics_every_epochs': 3, 'epochs': 35, 'backbone_lr': 1e-05}
import zipfile

run_id = RUN_IDS['teacher']
run_dir = ROOT / 'runs' / run_id
current = read_json(run_dir / 'run_summary.json')
initial = read_json(run_dir / 'initialization.json')
effective = read_json(run_dir / 'effective_dropout.json')
current_settings = read_json(run_dir / 'protocol.json')['settings']
reference_settings = dict(BASELINE_SETTINGS, backbone_dropout=0.1)
differences = {key:{'reference':reference_settings.get(key), 'new':current_settings.get(key)}
    for key in sorted(set(reference_settings) | set(current_settings))
    if reference_settings.get(key) != current_settings.get(key)}
initialization_matches = initial['initial_all_parameters_sha256'] == BASELINE_SUMMARY['initial_parameters_sha256']
dev = current['selected_dev']
current_row = {'run_id':run_id, 'best_epoch':current['best_epoch'],
    'train_f1':current['selected_train']['combined']['macro_f1'],
    'dev_f1':dev['combined']['macro_f1'],
    'CL_dev_f1':dev['domains']['CL']['macro_f1'], 'IT_dev_f1':dev['domains']['IT']['macro_f1'],
    'codabench_f1':None, 'training_completed':current['training_completed']}
reference_row = {k:v for k,v in BASELINE_SUMMARY.items() if k!='initial_parameters_sha256'}
display(pd.DataFrame([reference_row,current_row]))
print('Effective dropout:', effective)
print('Same initial parameter values as reference:', initialization_matches)
audit = {'experiment':'combined_heavier_regularization', 'reference':BASELINE_SUMMARY,
    'current':current_row, 'effective_dropout':effective,
    'initialization_matches':initialization_matches, 'setting_differences':differences,
    'dev_f1_delta':current_row['dev_f1']-BASELINE_SUMMARY['dev_f1'],
    'selection_basis':'Development macro-F1; hidden-test score of the new model is unknown.'}
write_json(run_dir / 'regularization_comparison.json', audit)
# Refresh only the small diagnostics archive. The resumable large backup already exists.
diagnostics = run_dir.parent / (run_id + '_diagnostics.zip')
with zipfile.ZipFile(diagnostics,'w',zipfile.ZIP_DEFLATED) as archive:
    for artifact in sorted(run_dir.rglob('*')):
        if artifact.is_file() and artifact.suffix in ('.json','.csv','.png','.py','.txt'):
            archive.write(artifact, run_id + '/' + str(artifact.relative_to(run_dir)))
print('Development macro-F1 change:', round(audit['dev_f1_delta'],6))
print('Training completed:', current['training_completed'])
print('A time-paused result remains provisional. Keep the earlier models and submissions.')
for suffix in ('_submission.zip','_diagnostics.zip','_backup.zip','_teacher_cache.zip'):
    output = run_dir.parent / (run_id + suffix)
    if output.exists():
        print(output, '| MiB', round(output.stat().st_size/2**20,2))
        display(FileLink(str(output)))
del CONTEXT
gc.collect()
torch.cuda.empty_cache()
print('GPU models released.')


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/cs781_phase3/runs/efficient_rr_teacher_heavyreg_001/run_summary.json'

## References

- Regularization methods. [Dropout](https://jmlr.org/papers/v15/srivastava14a.html) and [AdamW](https://arxiv.org/abs/1711.05101)
- Context-aware legal sentence classification. [Semantic Segmentation of Legal Documents via Rhetorical Roles](https://aclanthology.org/2022.nllp-1.13/)
- Learned sequence scores and Viterbi decoding. [Neural Architectures for Named Entity Recognition](https://aclanthology.org/N16-1030/)
